可以直接跳到 [Update 05/29](https://colab.research.google.com/drive/1M4_KBoDEZ0BYlI7N2MswYAXsc5jGKFyx#scrollTo=dyDTarFIUj6U&line=3&uniqifier=1) 直接執行

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import os

# Define potential paths for Ricky and HaoWen
paths = [
    '/content/drive/MyDrive/多變量分析/肺癌罹病率資料.csv',
    '/content/drive/MyDrive/1142課程/多變量分析/肺癌罹病率資料.csv'
]

csv_file_path = None
for path in paths:
    if os.path.exists(path):
        csv_file_path = path
        break

if csv_file_path:
    try:
        df = pd.read_csv(csv_file_path)
        print(f"CSV file loaded successfully from: {csv_file_path}")
        display(df.head())
    except Exception as e:
        print(f"An error occurred while reading the file: {e}")
else:
    print("Error: The file was not found in any of the specified paths. Please ensure Google Drive is mounted.")

In [ ]:
df_filtered = df[(df['性別'] == '全') &(df['縣市別'] != '全國') & (df['癌症別'] == '肺、支氣管及氣管')]
display(df_filtered)

In [ ]:
pivot_table_age_std = df_filtered.pivot_table(
    index='縣市別',
    columns='癌症診斷年',
    values='年齡標準化發生率  WHO 2000世界標準人口 (每10萬人口)',
    aggfunc='mean' # Assuming you want the mean if there are multiple entries
)
display(pivot_table_age_std)

In [ ]:
correlation_matrix = pivot_table_age_std.corr()
display(correlation_matrix.head())

Now, let's use hierarchical clustering to reorder the correlation matrix for better visualization in the heatmap.

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt
import seaborn as sns

# Perform hierarchical clustering
linked = sch.linkage(correlation_matrix, method='ward')

# Get the order of the clusters
cluster_order = sch.dendrogram(linked, no_plot=True)['leaves']

# Reorder the correlation matrix based on the clustering result
reordered_correlation_matrix = correlation_matrix.iloc[cluster_order, cluster_order]
display(reordered_correlation_matrix.head())

Finally, let's visualize the reordered correlation matrix using a heatmap.

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(reordered_correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Age-Standardized Incidence Rates (Ordered by Hclust)')
plt.show()

## Principal Component Analysis (PCA) on Correlation Matrix

Now, let's perform PCA on the `correlation_matrix` to reduce dimensionality and identify underlying patterns. We'll start by importing `PCA` from `sklearn.decomposition` and fitting it to our data.

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Initialize PCA. We will keep all components for the scree plot first.
pca = PCA()
pca.fit(correlation_matrix)

# Explained variance ratio
explained_variance_ratio = pca.explained_variance_ratio_

# Cumulative explained variance
cumulative_explained_variance = np.cumsum(explained_variance_ratio)

print("Explained Variance Ratio:")
print(explained_variance_ratio)
print("\nCumulative Explained Variance:")
print(cumulative_explained_variance)

### Scree Plot

A scree plot helps visualize the eigenvalues (explained variance) for each principal component, allowing us to determine the optimal number of components to retain.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker='o', linestyle='--')
plt.title('Scree Plot')
plt.xlabel('Number of Components')
plt.ylabel('Explained Variance Ratio')
plt.grid(True)
plt.show()

### Cumulative Explained Variance Plot

This plot shows the cumulative sum of explained variance, which helps in deciding how many principal components are needed to explain a certain percentage of the total variance.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_explained_variance) + 1), cumulative_explained_variance, marker='o', linestyle='--')
plt.axhline(y=0.8, color='r', linestyle='-', label='80% Cumulative Variance')
plt.title('Cumulative Explained Variance Plot')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.grid(True)
plt.legend()
plt.show()

### Biplot

A biplot visualizes the relationship between the observations (rows) and the variables (columns) in the PCA space. It helps in understanding how original variables contribute to the principal components and how observations are grouped.

In [ ]:
# For biplot, we usually consider the first two principal components
pca_2_components = PCA(n_components=2)
components = pca_2_components.fit_transform(correlation_matrix)
loadings = pca_2_components.components_.T * np.sqrt(pca_2_components.explained_variance_)

fig, ax = plt.subplots(figsize=(12, 12))
ax.scatter(components[:, 0], components[:, 1], alpha=0.5)

for i, feature in enumerate(correlation_matrix.columns):
    ax.arrow(0, 0, loadings[i, 0], loadings[i, 1], color='r', alpha=0.8)
    ax.text(loadings[i, 0] * 1.15, loadings[i, 1] * 1.15, feature, color='g', ha='center', va='center')

ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_title('Biplot of PCA on Correlation Matrix')
ax.grid(True)
plt.show()

## Cluster Analysis

We will now perform cluster analysis on the `pivot_table_age_std` to group similar `縣市別` (cities/counties) based on their age-standardized incidence rates over the years. We'll test different distance measures and clustering algorithms as requested.

### Data Preparation for Clustering

First, we need to handle any missing values in `pivot_table_age_std` and scale the data. Clustering algorithms are sensitive to missing values and the scale of features.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Ensure the data is numeric and handle potential NaNs
# For clustering, we need to ensure no NaNs. Let's impute with the mean of each column (year).
clustering_data = pivot_table_age_std.copy()

# Impute NaNs with the mean of each column
for col in clustering_data.columns:
    if clustering_data[col].isnull().any():
        clustering_data[col] = clustering_data[col].fillna(clustering_data[col].mean())

# Standardize the data. We want to cluster the rows (縣市別).
# So we scale features (years) for each observation (縣市別).
scaler = StandardScaler()
scaled_clustering_data = scaler.fit_transform(clustering_data)

# Convert back to DataFrame for easier inspection, keeping original index and columns
scaled_clustering_df = pd.DataFrame(scaled_clustering_data, index=clustering_data.index, columns=clustering_data.columns)
display(scaled_clustering_df.head())


### K-Means Clustering

We'll apply K-Means clustering. For demonstration, we'll choose a number of clusters (e.g., `k=4`). To visualize the clusters, we'll reduce the dimensionality using PCA to two components.

In [ ]:
from sklearn.decomposition import PCA

k = 4  # Let's start with 4 clusters for demonstration
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10) # n_init is set to 'auto' or explicit int in newer versions
kmeans_labels = kmeans.fit_predict(scaled_clustering_data)

# Add cluster labels to the original DataFrame
clustering_data['KMeans_Cluster'] = kmeans_labels
display(clustering_data.head())

# Visualize K-Means clusters using PCA for dimensionality reduction
pca_kmeans = PCA(n_components=2)
components_kmeans = pca_kmeans.fit_transform(scaled_clustering_data)

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=components_kmeans[:, 0],
    y=components_kmeans[:, 1],
    hue=kmeans_labels,
    palette='viridis',
    s=100,
    alpha=0.8
)
plt.title(f'K-Means Clustering (k={k}) - PCA Reduced')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.show()

### Hierarchical Clustering

Now, let's explore hierarchical clustering using different distance metrics and linkage methods. We'll use Pearson Correlation and Spearman Correlation as distance measures, and 'single', 'complete', and 'average' as linkage methods. For each combination, we'll display a dendrogram.

#### Hierarchical Clustering with Pearson Correlation Distance

In [ ]:
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler
import numpy as np

# Standardize data
scaler = StandardScaler()
scaled_clustering_data = scaler.fit_transform(pivot_table_age_std)

# Calculate Pearson correlation-based distance and handle non-finite values
pearson_distances = pdist(scaled_clustering_data, metric='correlation')
if not np.all(np.isfinite(pearson_distances)):
    pearson_distances = np.nan_to_num(pearson_distances, nan=1.0)

linkage_methods = ['single', 'complete', 'average']

# Ensure the font is set correctly for this cell as well
font_path = '/content/NotoSansTC-Regular.otf'
if os.path.exists(font_path):
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()

plt.figure(figsize=(18, 15))
for i, method in enumerate(linkage_methods):
    plt.subplot(3, 1, i + 1)
    linked = linkage(pearson_distances, method=method)
    dendrogram(linked,
               orientation='top',
               labels=pivot_table_age_std.index.tolist(),
               distance_sort='descending',
               show_leaf_counts=True,
               color_threshold=0.7 * max(linked[:,2]))
    plt.title(f'Hierarchical Clustering (Pearson - {method} Linkage)')
    plt.xlabel('County/City')
    plt.ylabel('Distance')

plt.tight_layout()
plt.show()

#### Hierarchical Clustering with Spearman Correlation Distance

In [ ]:
from scipy.stats import rankdata

# Define a custom function for Spearman correlation distance
def spearman_correlation_distance(u, v):
    # Calculate ranks for each vector u and v
    rank_u = rankdata(u)
    rank_v = rankdata(v)
    # Calculate Pearson correlation coefficient between the ranks
    corr_matrix = np.corrcoef(rank_u, rank_v)
    corr = corr_matrix[0, 1]

    # Handle NaN correlation (e.g., if one or both vectors are constant)
    if np.isnan(corr):
        # A common practice is to assign a neutral distance of 1.0 (equivalent to correlation 0)
        return 1.0
    return 1 - corr

# Calculate Spearman correlation-based distance using the custom metric
spearman_distances_condensed = pdist(scaled_clustering_data, metric=spearman_correlation_distance)

plt.figure(figsize=(18, 12))

for i, method in enumerate(linkage_methods):
    plt.subplot(3, 1, i + 1)
    linked = linkage(spearman_distances_condensed, method=method)
    dendrogram(linked,
               orientation='top',
               labels=clustering_data.index.tolist(),
               distance_sort='descending',
               show_leaf_counts=True,
               color_threshold=0.7 * max(linked[:,2])) # Adjust threshold for better visibility
    plt.title(f'Hierarchical Clustering (Spearman - {method} Linkage)')
    plt.xlabel('縣市別')
    plt.ylabel('Distance')

plt.tight_layout()
plt.show()

# Update 05/23

In [ ]:
import pandas as pd
import os

# Define potential paths for Ricky and HaoWen
paths = [
    '/content/drive/MyDrive/多變量分析/肺癌罹病率資料.csv',
    '/content/drive/MyDrive/1142課程/多變量分析/肺癌罹病率資料.csv'
]

csv_file_path = None
for path in paths:
    if os.path.exists(path):
        csv_file_path = path
        break

if csv_file_path:
    try:
        df = pd.read_csv(csv_file_path)
        print(f"CSV file loaded successfully from: {csv_file_path}")
        display(df.head())
    except Exception as e:
        print(f"An error occurred while reading the file: {e}")
else:
    print("Error: The file was not found in any of the specified paths. Please ensure Google Drive is mounted.")

In [ ]:
df.columns = [
    'Diagnosis_Year', 'Gender', 'City_County', 'Cancer_Type',
    'Age_Standardized_Incidence_Rate', 'Cancer_Incidence_Count',
    'Mean_Age', 'Median_Age', 'Crude_Incidence_Rate'
]
# Display the new column names and a preview of the data
print("New column names:", df.columns.tolist())
display(df.head())

In [ ]:
unique_cancer_types = df['Cancer_Type'].unique()
print(unique_cancer_types)

$$
標準化死亡率= {\sum(標準人口各年齡層之加權數×觀察族群各年齡層死亡率)}
$$

<br>
$\text{標準人口} = $ WHO 2000 年世界標準人口

In [ ]:
df_filtered = df[(df['Gender'] == '全') &(df['City_County'] != '全國') & (df['Cancer_Type'] == '肺、支氣管及氣管')]
display(df_filtered)

In [ ]:
pivot_table_age_std = df_filtered.pivot_table(
    index='City_County',
    columns='Diagnosis_Year',
    values='Age_Standardized_Incidence_Rate',
    aggfunc='mean' # Assuming you want the mean if there are multiple entries
)
display(pivot_table_age_std)

In [ ]:
corr_matrix_yr = pivot_table_age_std.corr()
display(corr_matrix_yr.head())

In [ ]:
corr_matrix_county = pivot_table_age_std.T.corr()
display(corr_matrix_county.head())

In [ ]:
corr_matrix_county.columns

Now, let's use hierarchical clustering to reorder the correlation matrix for better visualization in the heatmap.

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt
import seaborn as sns

# Perform hierarchical clustering
linked = sch.linkage(corr_matrix_yr, method='ward')

# Get the order of the clusters
cluster_order = sch.dendrogram(linked, no_plot=True)['leaves']

# Reorder the correlation matrix based on the clustering result
reordered_corr_matrix_yr = corr_matrix_yr.iloc[cluster_order, cluster_order]
display(reordered_corr_matrix_yr.head())

In [ ]:
# Perform hierarchical clustering
linked = sch.linkage(corr_matrix_county, method='ward')

# Get the order of the clusters
cluster_order = sch.dendrogram(linked, no_plot=True)['leaves']

# Reorder the correlation matrix based on the clustering result
reordered_corr_matrix_county = corr_matrix_county.iloc[cluster_order, cluster_order]
display(reordered_corr_matrix_county.head())

Finally, let's visualize the reordered correlation matrix using a heatmap.

In [ ]:
!wget -q -O /content/NotoSansTC-Regular.otf https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import os

# Register the font file specifically
font_path = '/content/NotoSansTC-Regular.otf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()

plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# Re-draw the heatmap with fixed range
plt.figure(figsize=(12, 10))
sns.heatmap(reordered_corr_matrix_yr, annot=False, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Age-Standardized Incidence Rates by Years (Ordered by Hclust)')
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(reordered_corr_matrix_county, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Age-Standardized Incidence Rates by Counties (Ordered by Hclust)')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure Chinese font is applied
if 'prop' in locals() or 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

# Calculate base font sizes (increasing by 20%)
base_size = 12 * 1.1
title_size = 16 * 1.1
label_size = 14 * 1.1

plt.figure(figsize=(14, 8))
sns.lineplot(data=df_filtered, x='Diagnosis_Year', y='Age_Standardized_Incidence_Rate', hue='City_County')

plt.title('肺癌—年齡標準化發生率—按縣市與年度', fontsize=title_size)
plt.xlabel('年', fontsize=label_size)
plt.ylabel('年齡標準化發生率 (每10萬人口)', fontsize=label_size)
plt.legend(title='縣市別', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=base_size, title_fontsize=label_size)
plt.xticks(fontsize=base_size)
plt.yticks(fontsize=base_size)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the Chinese font is applied for the plot
if 'prop' in locals() or 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

plt.figure(figsize=(14, 8))
sns.lineplot(data=df_filtered, x='Diagnosis_Year', y='Median_Age', hue='City_County')

plt.title('Median Age of Cancer Patients by Year and City/County')
plt.xlabel('Year')
plt.ylabel('Median Age')
plt.legend(title='City/County', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

## Principal Component Analysis (PCA) on Correlation Matrix (County)

Now, let's perform PCA on the `correlation_matrix` to reduce dimensionality and identify underlying patterns. We'll start by importing `PCA` from `sklearn.decomposition` and fitting it to our data.

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Initialize PCA. We will keep all components for the scree plot first.
pca = PCA()
pca.fit(corr_matrix_county)

# Explained variance ratio
explained_variance_ratio = pca.explained_variance_ratio_

# Cumulative explained variance
cumulative_explained_variance = np.cumsum(explained_variance_ratio)

print("Explained Variance Ratio:")
print(explained_variance_ratio)
print("\nCumulative Explained Variance:")
print(cumulative_explained_variance)

### Scree Plot

A scree plot helps visualize the eigenvalues (explained variance) for each principal component, allowing us to determine the optimal number of components to retain.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker='o', linestyle='--')
plt.title('Scree Plot')
plt.xlabel('Number of Components')
plt.ylabel('Explained Variance Ratio')
plt.grid(True)
plt.show()

### Cumulative Explained Variance Plot

This plot shows the cumulative sum of explained variance, which helps in deciding how many principal components are needed to explain a certain percentage of the total variance.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_explained_variance) + 1), cumulative_explained_variance, marker='o', linestyle='--')
plt.axhline(y=0.8, color='r', linestyle='-', label='80% Cumulative Variance')
plt.title('Cumulative Explained Variance Plot')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.grid(True)
plt.legend()
plt.show()

### Biplot

A biplot visualizes the relationship between the observations (rows) and the variables (columns) in the PCA space. It helps in understanding how original variables contribute to the principal components and how observations are grouped.

In [ ]:
!pip install adjustText

In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.font_manager as fm

# Force installation and ensure path refresh
!pip install -q adjustText

import site
import importlib
importlib.reload(site)

try:
    from adjustText import adjust_text
except ImportError:
    from adjust_text import adjust_text

# Perform PCA on the county correlation matrix
pca_2_components = PCA(n_components=2)
components = pca_2_components.fit_transform(corr_matrix_county)
loadings = pca_2_components.components_.T * np.sqrt(pca_2_components.explained_variance_)

# Calculate scaling factor to balance components and loadings visually
scale_factor = 2.0

fig, ax = plt.subplots(figsize=(12, 12))

# Ensure Chinese font is applied
font_path = '/content/NotoSansTC-Regular.otf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()

# Plot the observations (Counties represented by their PC scores)
ax.scatter(components[:, 0], components[:, 1], alpha=0.3, color='blue', label='縣市得分')

texts = []
# Plot the loadings (Arrows for years)
for i, feature in enumerate(corr_matrix_county.columns):
    dx = loadings[i, 0] * scale_factor
    dy = loadings[i, 1] * scale_factor
    ax.arrow(0, 0, dx, dy, color='r', alpha=0.4, head_width=0.01)
    texts.append(ax.text(dx, dy, feature, color='darkgreen', weight='bold', fontsize=12))

# Use adjust_text to fix overlapping labels
adjust_text(texts, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

ax.set_xlabel('Principal Component 1', fontsize=14)
ax.set_ylabel('Principal Component 2', fontsize=14)
ax.set_title('PCA Biplot - 縣市特徵分析 (重疊修正版)', fontsize=16)
ax.grid(True, linestyle='--', alpha=0.6)
plt.axhline(0, color='black', linewidth=1)
plt.axvline(0, color='black', linewidth=1)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np
import os
import matplotlib.font_manager as fm

# Perform PCA on county correlation matrix
pca_2_components = PCA(n_components=2)
components = pca_2_components.fit_transform(corr_matrix_county)
loadings = pca_2_components.components_.T * np.sqrt(pca_2_components.explained_variance_)

# Scaling factor for arrows
scale_factor = 1.5

fig, ax = plt.subplots(figsize=(10, 10))

# Font configuration for Chinese characters
font_path = '/content/NotoSansTC-Regular.otf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()

# Plot score points (Counties)
ax.scatter(components[:, 0], components[:, 1], alpha=0.5, color='blue', label='縣市得分')

# Plot loading arrows (Years) and add labels if they match target counties
target_counties = ['金門縣', '連江縣']
for i, county_name in enumerate(corr_matrix_county.columns):
    dx = loadings[i, 0] * scale_factor
    dy = loadings[i, 1] * scale_factor
    ax.arrow(0, 0, dx, dy, color='r', alpha=0.4, head_width=0.02)

    if county_name in target_counties:
        ax.text(dx * 1.15, dy * 1.15, county_name, fontsize=14, color='darkred', weight='bold')

# Also label the corresponding data points (scores)
for i, county_name in enumerate(corr_matrix_county.index):
    if county_name in target_counties:
        ax.text(components[i, 0] + 0.015, components[i, 1] + 0.015, f"{county_name} (Score)",
                fontsize=10, color='blue', alpha=0.7)

ax.set_xlabel('Principal Component 1', fontsize=12)
ax.set_ylabel('Principal Component 2', fontsize=12)
ax.set_title('PCA Biplot - 縣市', fontsize=15)
ax.grid(True, linestyle='--', alpha=0.5)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
plt.show()

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Re-calculating PCA components and explained variance ratio for corr_matrix_county
# This is needed for cell WRFIllJFiL6F
pca_2_components = PCA(n_components=2)
components = pca_2_components.fit_transform(corr_matrix_county)
explained_variance_ratio = pca_2_components.explained_variance_ratio_

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering

# 1. 取得縣市名稱列表
county_names = corr_matrix_county.index.tolist()

# 2. 利用你之前的階層分群設定，假設我們切成 4 群
hclust = AgglomerativeClustering(n_clusters=4, linkage='ward')
cluster_labels = hclust.fit_predict(corr_matrix_county)

# 3. 建立 PCA 畫圖用的 DataFrame
pca_df = pd.DataFrame({
    'PC1': components[:, 0],
    'PC2': components[:, 1],
    'County': county_names,
    'Cluster': [f'Cluster {c}' for c in cluster_labels]
})

# 4. 開始畫圖
plt.figure(figsize=(14, 10))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Cluster', palette='Set1', s=150, alpha=0.8)

# Determine the font properties to use for annotations
annotation_font_prop = None
if 'prop' in globals() and prop is not None: # Defaulting to Noto Sans TC if available
    annotation_font_prop = prop
elif 'prop_serif' in globals() and prop_serif is not None: # Fallback to Noto Serif TC
    annotation_font_prop = prop_serif

# 加註縣市名稱
for i, txt in enumerate(pca_df['County']):
    # Corrected argument: 'fontproperties' instead of 'prop'
    plt.annotate(txt, (pca_df['PC1'].iloc[i], pca_df['PC2'].iloc[i]),
                 textcoords="offset points", xytext=(5,5), ha='center', fontsize=11,
                 fontproperties=annotation_font_prop) # Use fontproperties here

plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.axvline(0, color='grey', linestyle='--', linewidth=0.8)
plt.title('PCA Biplot of Taiwan Counties based on Lung Cancer Trends', fontsize=16)
plt.xlabel(f'PC1 ({explained_variance_ratio[0]*100:.2f}%)', fontsize=12)
plt.ylabel(f'PC2 ({explained_variance_ratio[1]*100:.2f}%)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(title='Hclust Groups')
plt.show()

In [ ]:
loadings

In [ ]:
components

## 排除連江縣做 PCA

In [ ]:
# Check for rows (counties) with NaN values
rows_with_nan = pivot_table_age_std[pivot_table_age_std.isna().any(axis=1)].index.tolist()

# Check for columns (years) with NaN values
cols_with_nan = pivot_table_age_std.columns[pivot_table_age_std.isna().any(axis=0)].tolist()

print(f"Rows (Counties) with NaN values: {rows_with_nan}")
print(f"Columns (Years) with NaN values: {cols_with_nan}")

# Display the exact locations (Row, Column) where values are missing
if rows_with_nan or cols_with_nan:
    print("\nDetailed NaN locations:")
    nan_locations = pivot_table_age_std.isna().stack()
    nan_locations = nan_locations[nan_locations].index.tolist()
    for row, col in nan_locations:
        print(f"- County: {row}, Year: {col}")
else:
    print("No NaN values found in the table.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering

# ==========================================
# 1. 資料處理 (Drop NaNs) 與標準化
# ==========================================
# 排除含有缺失值的連江縣
pivot_table_cleaned = pivot_table_age_std.drop(index='連江縣')

scaler = StandardScaler()
scaled_data = scaler.fit_transform(pivot_table_cleaned)

# ==========================================
# 2. 執行 PCA
# ==========================================
pca_full = PCA()
pca_full.fit(scaled_data)

print("--- PCA 變異解釋量分析 (排除連江縣) ---")
print("各主成分解釋變異比 (前5個):", pca_full.explained_variance_ratio_[:5])
print("累積解釋變異比 (前5個):", np.cumsum(pca_full.explained_variance_ratio_)[:5])
print("-" * 30)

# 取前兩個主成分做後續分析
n_comps = 2
pca_2d = PCA(n_components=n_comps)
scores_matrix = pca_2d.fit_transform(scaled_data) # 縣市的新座標

# 計算 Loadings (年份與 PC 的相關性)
loadings_matrix = pca_2d.components_.T * np.sqrt(pca_2d.explained_variance_)

# ==========================================
# 3. 縣市分群 (Hclust)
# ==========================================
hclust = AgglomerativeClustering(n_clusters=4, linkage='ward')
cluster_labels = hclust.fit_predict(scaled_data)

# 建立 Scores DataFrame
counties = pivot_table_cleaned.index
df_scores = pd.DataFrame({
    'PC1_Score': scores_matrix[:, 0],
    'PC2_Score': scores_matrix[:, 1],
    'County': counties,
    'Cluster': [f'Group {c+1}' for c in cluster_labels]
})

# 建立 Loadings DataFrame (年份)
years = pivot_table_cleaned.columns
df_loadings = pd.DataFrame({
    'Year': years,
    'PC1_Loading': loadings_matrix[:, 0],
    'PC2_Loading': loadings_matrix[:, 1]
})

# ==========================================
# 4. 視覺化：縣市得分圖 (Scores Plot)
# ==========================================
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df_scores, x='PC1_Score', y='PC2_Score', hue='Cluster', palette='Set1', s=150, alpha=0.8)

# 加上縣市名稱標籤
if 'prop' in locals() or 'prop' in globals():
    font_name = prop.get_name()
else:
    font_name = None

for i, txt in enumerate(df_scores['County']):
    plt.annotate(txt, (df_scores['PC1_Score'].iloc[i], df_scores['PC2_Score'].iloc[i]),
                 textcoords="offset points", xytext=(0,10), ha='center', fontsize=11, fontname=font_name)

plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.axvline(0, color='grey', linestyle='--', linewidth=0.8)
plt.title('County Scores Plot (Excluding Lienchiang County)', fontsize=14)
plt.xlabel(f"PC1 Score ({pca_full.explained_variance_ratio_[0]*100:.2f}%)", fontsize=12)
plt.ylabel(f"PC2 Score ({pca_full.explained_variance_ratio_[1]*100:.2f}%)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(title='Hierarchical Clusters')
plt.tight_layout()
plt.show()

# ==========================================
# 5. 視覺化：年份負荷量隨時間演變圖
# ==========================================
plt.figure(figsize=(12, 5))
plt.plot(df_loadings['Year'], df_loadings['PC1_Loading'], marker='o', label='PC1 Loading', color='navy', linewidth=2)
plt.plot(df_loadings['Year'], df_loadings['PC2_Loading'], marker='s', label='PC2 Loading', color='crimson', linewidth=2)
plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.title('Loadings over Time (Excluding Lienchiang County)', fontsize=14)
plt.xlabel('Diagnosis Year', fontsize=12)
plt.ylabel('Loading Value', fontsize=12)
plt.xticks(df_loadings['Year'][::4], rotation=45)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### PCA 變異解釋量分析視覺化
透過 Scree Plot 與累積變異圖，我們可以評估降維的效果。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 取得解釋變異量比率
exp_var_ratio = pca_full.explained_variance_ratio_
cum_var_ratio = np.cumsum(exp_var_ratio)

fig, ax1 = plt.subplots(figsize=(10, 6))

# 1. 繪製 Scree Plot (左軸)
color = 'tab:blue'
ax1.set_xlabel('Principal Component Index')
ax1.set_ylabel('Explained Variance Ratio', color=color)
ax1.plot(range(1, len(exp_var_ratio) + 1), exp_var_ratio, marker='o', linestyle='--', color=color, label='Individual')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(range(1, len(exp_var_ratio) + 1, 2))
ax1.set_ylim(-.1, 1.1)  # 設定左軸範圍為 (0, 1)

# 2. 繪製 Cumulative Variance Plot (右軸)
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Cumulative Explained Variance', color=color)
ax2.plot(range(1, len(cum_var_ratio) + 1), cum_var_ratio, marker='s', linestyle='-', color=color, label='Cumulative')
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(-.1, 1.1)  # 設定右軸範圍為 (0, 1)

# 標示 80% 門檻線
ax2.axhline(y=0.8, color='gray', linestyle=':', alpha=0.7)
ax2.text(len(cum_var_ratio)*0.7, 0.82, '80% Threshold', color='gray', fontsize=10)

plt.title('PCA Scree Plot & Cumulative Explained Variance (Excluding Lienchiang)')
fig.tight_layout()
plt.grid(True, alpha=0.2)
plt.show()

### PC1 ~ PC4 負荷量 (Loadings) 分析
我們提取前四個主成分（累積解釋變異量 > 80%），並觀察各年份在這些成分上的負荷量。這有助於定義每個 PC 所代表的「時間特徵」。

In [ ]:
# 1. 提取前四個主成分的 Loadings
pca_4 = PCA(n_components=4)
pca_4.fit(scaled_data)
loadings_4 = pca_4.components_.T * np.sqrt(pca_4.explained_variance_)

# 建立 DataFrame
years = pivot_table_cleaned.columns
df_loadings_4 = pd.DataFrame(loadings_4, index=years, columns=['PC1', 'PC2', 'PC3', 'PC4'])

# 2. 繪製 Loadings 熱圖
plt.figure(figsize=(10, 12))
sns.heatmap(df_loadings_4, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('PCA Loadings Heatmap (PC1 - PC4)', fontsize=14)
plt.ylabel('Year')
plt.show()

In [ ]:
print(loadings_4)

### 縣市得分圖 (Scores Plot): PC3 vs PC4
除了主要的 PC1/PC2，我們觀察 PC3 與 PC4 是否能區分出特定的縣市群體。

In [ ]:
# 計算縣市得分
scores_4 = pca_4.transform(scaled_data)
df_scores_4 = pd.DataFrame(scores_4, index=pivot_table_cleaned.index, columns=['PC1', 'PC2', 'PC3', 'PC4'])

# 繪製 PC3 vs PC4
plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_scores_4, x='PC3', y='PC4', s=100, color='darkorange', alpha=0.7)

# 標記縣市名稱
for i, county in enumerate(df_scores_4.index):
    plt.annotate(county, (df_scores_4['PC3'].iloc[i], df_scores_4['PC4'].iloc[i]),
                 textcoords="offset points", xytext=(0,10), ha='center', fontsize=10, fontname=font_name)

plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.axvline(0, color='grey', linestyle='--', linewidth=0.8)
plt.title('County Scores Plot: PC3 vs PC4', fontsize=14)
plt.xlabel(f'PC3 ({pca_4.explained_variance_ratio_[2]*100:.2f}%)')
plt.ylabel(f'PC4 ({pca_4.explained_variance_ratio_[3]*100:.2f}%)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 確保已剔除連江縣，並建立包含分群標籤的 DataFrame
df_plot_raw = pivot_table_age_std.drop(index='連江縣', errors='ignore').copy()

# 假設 df_scores 是我們上一步做好的 DataFrame (內含 County 和 Cluster 欄位)
# 將分群資訊 mapping 回原始的 pivot table
df_plot_raw = df_plot_raw.reset_index()
df_plot_raw = df_plot_raw.merge(df_scores[['County', 'Cluster']], left_on='City_County', right_on='County')

# 2. 將資料由寬轉長 (Melt)，方便 Seaborn 畫折線圖
df_long = df_plot_raw.drop(columns=['County']).melt(
    id_vars=['City_County', 'Cluster'],
    var_name='Year',
    value_name='Incidence_Rate'
)
df_long['Year'] = df_long['Year'].astype(int)

# 3. 開始畫圖：利用 FacetGrid 依群組分開畫，或者全部畫在同一張
plt.figure(figsize=(14, 8))

# 畫出每個縣市自己的線 (estimator=None, units='City_County')
sns.lineplot(
    data=df_long,
    x='Year',
    y='Incidence_Rate',
    hue='Cluster',
    units='City_County',
    estimator=None,
    palette='Set1',
    alpha=0.4,
    linewidth=1.5
)

# 畫出各群組的「平均趨勢粗線」作為對比
sns.lineplot(
    data=df_long,
    x='Year',
    y='Incidence_Rate',
    hue='Cluster',
    palette='Set1',
    linewidth=4,
    legend=False
)

# 特別標記出 2003 年分水嶺
plt.axvline(2003, color='black', linestyle=':', linewidth=2, label='2003 Cancer Control Act')

plt.title('Individual County Lung Cancer Trends Grouped by PCA Clusters', fontsize=16)
plt.xlabel('Diagnosis Year', fontsize=12)
plt.ylabel('Age-Standardized Incidence Rate (per 100k)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cluster Analysis

We will now perform cluster analysis on the `pivot_table_age_std` to group similar `縣市別` (cities/counties) based on their age-standardized incidence rates over the years. We'll test different distance measures and clustering algorithms as requested.

### K-Means Clustering

We'll apply K-Means clustering. For demonstration, we'll choose a number of clusters (e.g., `k=4`). To visualize the clusters, we'll reduce the dimensionality using PCA to two components.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 1. Perform K-means with k=4
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(scaled_data)

# 2. Map numeric labels to A, B, C, D
cluster_map = {0: 'Group A', 1: 'Group B', 2: 'Group C', 3: 'Group D'}
encoded_labels = [cluster_map[label] for label in kmeans_labels]

# 3. Create plotting DataFrame
counties_names = pivot_table_cleaned.index
pca_kmeans = PCA(n_components=2)
components_kmeans = pca_kmeans.fit_transform(scaled_data)

df_kmeans_plot = pd.DataFrame({
    'PC1': components_kmeans[:, 0],
    'PC2': components_kmeans[:, 1],
    'County': counties_names,
    'Cluster': encoded_labels
})

# 4. Visualization with high-contrast palette
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df_kmeans_plot,
    x='PC1',
    y='PC2',
    hue='Cluster',
    hue_order=['Group A', 'Group B', 'Group C', 'Group D'],
    palette='bright',
    s=200,
    alpha=0.9,
    edgecolor='black'
)

# Annotate county names
for i in range(len(df_kmeans_plot)):
    plt.annotate(
        df_kmeans_plot['County'].iloc[i],
        (df_kmeans_plot['PC1'].iloc[i], df_kmeans_plot['PC2'].iloc[i]),
        textcoords='offset points',
        xytext=(0, 12),
        ha='center',
        fontsize=11,
        fontname=prop.get_name() if 'prop' in locals() else None,
        bbox=dict(boxstyle='round,pad=0.2', fc='yellow', alpha=0.3)
    )

plt.title(f'K-Means Clustering (k={k}) - High Contrast View', fontsize=16)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.grid(True, alpha=0.3)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.legend(title='Clusters', loc='best')
plt.show()

# 5. Print the list of counties per cluster
print("\n--- Cluster Assignments ---")
for group in ['Group A', 'Group B', 'Group C', 'Group D']:
    members = df_kmeans_plot[df_kmeans_plot['Cluster'] == group]['County'].tolist()
    print(f"{group}: {', '.join(members)}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 1. Perform K-means with k=2
k = 2
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(scaled_data)

# 2. Map numeric labels to Group A, B
cluster_map = {0: 'Group A', 1: 'Group B'}
encoded_labels = [cluster_map[label] for label in kmeans_labels]

# 3. Create plotting DataFrame
counties_names = pivot_table_cleaned.index
pca_kmeans = PCA(n_components=2)
components_kmeans = pca_kmeans.fit_transform(scaled_data)

df_kmeans_plot = pd.DataFrame({
    'PC1': components_kmeans[:, 0],
    'PC2': components_kmeans[:, 1],
    'County': counties_names,
    'Cluster': encoded_labels
})

# 4. Visualization with high-contrast palette
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df_kmeans_plot,
    x='PC1',
    y='PC2',
    hue='Cluster',
    hue_order=['Group A', 'Group B'],
    palette='bright',
    s=200,
    alpha=0.9,
    edgecolor='black'
)

# Annotate county names
for i in range(len(df_kmeans_plot)):
    plt.annotate(
        df_kmeans_plot['County'].iloc[i],
        (df_kmeans_plot['PC1'].iloc[i], df_kmeans_plot['PC2'].iloc[i]),
        textcoords='offset points',
        xytext=(0, 12),
        ha='center',
        fontsize=11,
        fontname=prop.get_name() if 'prop' in locals() else None,
        bbox=dict(boxstyle='round,pad=0.2', fc='yellow', alpha=0.3)
    )

plt.title(f'K-Means Clustering (k={k}) - High Contrast View', fontsize=16)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.grid(True, alpha=0.3)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.legend(title='Clusters', loc='best')
plt.show()

# 5. Print the list of counties per cluster
print(f"\n--- Cluster Assignments (k={k}) ---")
for group in ['Group A', 'Group B']:
    members = df_kmeans_plot[df_kmeans_plot['Cluster'] == group]['County'].tolist()
    print(f"{group}: {', '.join(members)}")

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import os
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from sklearn.cluster import KMeans

# 1. Load the shapefile
shapefile_path = '/content/COUNTY_MOI_1140318.shp'
taiwan_map = gpd.read_file(shapefile_path).to_crs(epsg=4326)
county_col = 'COUNTYNAME' if 'COUNTYNAME' in taiwan_map.columns else [c for c in taiwan_map.columns if 'NAME' in c.upper()][0]

# 2. Font configuration
font_path = '/content/NotoSansTC-Regular.otf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()

# 3. Define Plotting Helper
def get_cluster_df(k_val):
    kmeans = KMeans(n_clusters=k_val, random_state=42, n_init=10)
    labels = kmeans.fit_predict(scaled_data)
    cluster_map = {0: 'Group A', 1: 'Group B', 2: 'Group C', 3: 'Group D'}
    df_res = pd.DataFrame({'County': pivot_table_cleaned.index, 'Cluster': [cluster_map[l] for l in labels]})
    df_res['County_Match'] = df_res['County'].str.replace('台', '臺')
    return df_res

# 4. Rendering 1x2 Subplots
fig, axes = plt.subplots(1, 2, figsize=(20, 10), dpi=100)
cluster_colors = {'Group A': '#6EA1FF', 'Group B': '#FFB366', 'Group C': '#82E0AA', 'Group D': '#F1948A'}

for ax, k in zip(axes, [2, 4]):
    df_k = get_cluster_df(k)
    merged = taiwan_map.merge(df_k, left_on=county_col, right_on='County_Match', how='left')
    merged['plot_color'] = merged['Cluster'].map(cluster_colors).fillna('#e0e0e0')

    merged.plot(ax=ax, color=merged['plot_color'], edgecolor='gray', linewidth=0.5)
    ax.set_xlim(118.0, 122.5)
    ax.set_ylim(21.5, 26.5)
    ax.set_title(f'K-Means Clustering (k={k})', fontsize=16)
    ax.set_axis_off()

    # Custom legend
    legend_elements = [Line2D([0], [0], marker='o', color='w', label=list(cluster_colors.keys())[i],
                              markerfacecolor=list(cluster_colors.values())[i], markersize=10) for i in range(k)]
    ax.legend(handles=legend_elements, loc='upper left', title='Clusters')

    # Labels
    for idx, row in merged.iterrows():
        if pd.notnull(row['Cluster']):
            centroid = row['geometry'].centroid
            ax.annotate(text=row['County'], xy=(centroid.x, centroid.y), ha='center', fontsize=8,
                        bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6, ec='none'))

plt.suptitle('Taiwan Lung Cancer Spatial Trends: k=2 vs k=4', fontsize=20, y=0.95)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Prepare Group Data for both k=2 and k=4
# K=2 Grouping
k2_group_a = df_kmeans_plot[df_kmeans_plot['Cluster'] == 'Group A']['County'].tolist()
k2_group_b = df_kmeans_plot[df_kmeans_plot['Cluster'] == 'Group B']['County'].tolist()

# K=4 Grouping (Calculated dynamically)
k4_kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
k4_labels = k4_kmeans.fit_predict(scaled_data)
cluster_map_4 = {0: 'Group A', 1: 'Group B', 2: 'Group C', 3: 'Group D'}
df_k4 = pd.DataFrame({'County': pivot_table_cleaned.index, 'Cluster': [cluster_map_4[l] for l in k4_labels]})

# 2. Process Data for Plotting
df_long_k2 = df_long.copy()
df_long_k2['Macro_Group'] = df_long_k2['City_County'].apply(lambda x: 'Group A' if x in k2_group_a else 'Group B')
df_group_means_k2 = df_long_k2.groupby(['Year', 'Macro_Group'])['Incidence_Rate'].mean().reset_index()

df_long_k4 = df_long.merge(df_k4, left_on='City_County', right_on='County', suffixes=('', '_k4'))
df_group_means_k4 = df_long_k4.groupby(['Year', 'Cluster_k4'])['Incidence_Rate'].mean().reset_index()

# 3. Create 2x1 Subplot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), sharex=True)

# Top Plot: k=2
sns.lineplot(data=df_group_means_k2, x='Year', y='Incidence_Rate', hue='Macro_Group',
             palette=['#6EA1FF', '#FFB366'], linewidth=3, ax=ax1)
ax1.set_title('Annual Average Incidence: Macro Groups (k=2)', fontsize=14)
ax1.set_ylabel('Mean Rate (per 100k)')
ax1.grid(True, alpha=0.3)

# Bottom Plot: k=4
sns.lineplot(data=df_group_means_k4, x='Year', y='Incidence_Rate', hue='Cluster_k4',
             palette='bright', linewidth=3, ax=ax2)
ax2.set_title('Annual Average Incidence: Detailed Groups (k=4)', fontsize=14)
ax2.set_ylabel('Mean Rate (per 100k)')
ax2.set_xlabel('Diagnosis Year')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. Summary Stats
print("--- Growth Summary (1979-2022) ---")
for group in df_group_means_k2['Macro_Group'].unique():
    subset = df_group_means_k2[df_group_means_k2['Macro_Group'] == group]
    start, end = subset.iloc[0]['Incidence_Rate'], subset.iloc[-1]['Incidence_Rate']
    print(f"{group} (k=2): {start:.2f} -> {end:.2f} ({((end-start)/start)*100:.1f}%)")

### Hierarchical Clustering

Now, let's explore hierarchical clustering using different distance metrics and linkage methods. We'll use Pearson Correlation and Spearman Correlation as distance measures, and 'single', 'complete', and 'average' as linkage methods. For each combination, we'll display a dendrogram.

#### Hierarchical Clustering with Pearson Correlation Distance

In [ ]:
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

# Standardize data - Handling NaNs (e.g., Lienchiang County)
clustering_data_clean = pivot_table_age_std.dropna()

scaler = StandardScaler()
scaled_clustering_data = scaler.fit_transform(clustering_data_clean)

# Calculate Pearson correlation-based distance
pearson_distances = pdist(scaled_clustering_data, metric='correlation')
if not np.all(np.isfinite(pearson_distances)):
    pearson_distances = np.nan_to_num(pearson_distances, nan=1.0)

# Linkage methods to visualize
linkage_methods = ['single', 'complete', 'average', 'ward']

# Font configuration for Chinese labels
font_path = '/content/NotoSansTC-Regular.otf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()

# Generate four separate plots
for method in linkage_methods:
    plt.figure(figsize=(12, 6))
    linked = linkage(pearson_distances, method=method)

    dendrogram(linked,
               orientation='top',
               labels=clustering_data_clean.index.tolist(),
               distance_sort='descending',
               show_leaf_counts=True,
               color_threshold=0.7 * max(linked[:, 2]))

    plt.title(f'Hierarchical Clustering (Pearson - {method.capitalize()} Linkage)', fontsize=14)
    plt.xlabel('County/City')
    plt.ylabel('Distance')
    plt.tight_layout()
    plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import matplotlib.pyplot as plt
import pandas as pd

# 設定分群數
k_clusters = 4
linkage_methods = ['single', 'complete', 'average']
counties = clustering_data_clean.index.tolist()

plt.figure(figsize=(18, 15))

for i, method in enumerate(linkage_methods):
    # 1. 執行階層分群
    linked = linkage(pearson_distances, method=method)

    # 2. 繪製樹狀圖
    plt.subplot(3, 1, i + 1)
    dendrogram(linked,
               orientation='top',
               labels=counties,
               distance_sort='descending',
               show_leaf_counts=True,
               color_threshold=0.7 * max(linked[:,2]))
    plt.title(f'Hierarchical Clustering (Pearson - {method.capitalize()} Linkage)')
    plt.xlabel('County/City')
    plt.ylabel('Distance')

    # 3. 取得分群結果並列印
    cluster_labels = fcluster(linked, k_clusters, criterion='maxclust')
    df_clusters = pd.DataFrame({'County': counties, 'Cluster': cluster_labels})

    print(f"\n--- {method.capitalize()} Linkage Cluster Results (k={k_clusters}) ---")
    for c in range(1, k_clusters + 1):
        members = df_clusters[df_clusters['Cluster'] == c]['County'].tolist()
        print(f"Group {c}: {', '.join(members)}")

plt.tight_layout()
plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import pandas as pd

# 1. 定義參數
methods = ['single', 'complete', 'average', 'ward']
k_fixed = 4
sil_results = []

# 2. 迴圈計算各個 linkage method 的 silhouette score
for method in methods:
    # 使用 AgglomerativeClustering 執行分群
    clusterer = AgglomerativeClustering(n_clusters=k_fixed, linkage=method)
    labels = clusterer.fit_predict(scaled_clustering_data)

    # 計算輪廓分數
    score = silhouette_score(scaled_clustering_data, labels)
    sil_results.append({
        'Linkage Method': method.capitalize(),
        'Clusters (k)': k_fixed,
        'Silhouette Score': round(score, 4)
    })

# 3. 以 DataFrame 輸出結果
df_sil_comparison = pd.DataFrame(sil_results)
print("--- Silhouette Score Comparison (k=4) ---")
display(df_sil_comparison)

# 4. 保留原有的 Elbow 趨勢分析 (以表現較佳的 Ward 為例)
K_range = range(2, 10)
sil_scores_trend = []

for k in K_range:
    clusterer = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = clusterer.fit_predict(scaled_clustering_data)
    sil_scores_trend.append(silhouette_score(scaled_clustering_data, labels))

# 繪製趨勢圖
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.plot(K_range, sil_scores_trend, marker='o', color='tab:blue', label='Ward Silhouette Score')
plt.title('Silhouette Score Trend (Ward Linkage)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Score')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 初始化參數
methods = ['single', 'complete', 'average', 'ward']
k_range = range(2, 11)
results = []

# 2. 巢狀迴圈計算分數
for k in k_range:
    for method in methods:
        clusterer = AgglomerativeClustering(n_clusters=k, linkage=method)
        labels = clusterer.fit_predict(scaled_clustering_data)
        score = silhouette_score(scaled_clustering_data, labels)
        results.append({
            'k': k,
            'Method': method.capitalize(),
            'Silhouette Score': score
        })

# 3. 轉換為 DataFrame
df_comparison = pd.DataFrame(results)

# 4. 建立樞紐表以便觀察
df_pivot = df_comparison.pivot(index='k', columns='Method', values='Silhouette Score')
print("--- Silhouette Score Comparison (k=2 to 10) ---")
display(df_pivot)

# 5. 繪製趨勢圖
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_comparison, x='k', y='Silhouette Score', hue='Method', marker='o')
plt.title('Silhouette Score Trends by Linkage Methods', fontsize=14)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(list(k_range))
plt.legend(title='Linkage Method')
plt.show()

In [ ]:
print(df_pivot)

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.lines import Line2D
import pandas as pd

# 1. Prepare clustering setup
methods = ['single', 'complete', 'average', 'ward']
# k_values = [4,4,4,5] VS [2,2,2,2]
k_values = [4,4,4,5] # Number of clusters for each method
counties_cleaned = pivot_table_cleaned.index

# 2. Setup figure for 2x2 comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 16), dpi=100)
axes = axes.flatten()

# Define consistent color mapping (Extended to 5 for Ward linkage)
color_map = {
    1: '#00FFFF', # Cyan
    2: '#FFFF00', # Yellow
    3: '#FF00FF', # Magenta
    4: '#00FF00', # Lime
    5: '#FF8000'  # Orange
}

for i, method in enumerate(methods):
    current_k = k_values[i]
    # Perform linkage
    Z = linkage(pearson_distances, method=method)
    # Form flat clusters
    labels = fcluster(Z, current_k, criterion='maxclust')

    # Map labels to the counties
    df_temp = pd.DataFrame({'County': counties_cleaned, 'Cluster': labels})
    df_temp['County_Match'] = df_temp['County'].str.replace('台', '臺')

    # Merge with the map (taiwan_map already in WGS84)
    merged_temp = taiwan_map.merge(df_temp, left_on='COUNTYNAME', right_on='County_Match', how='left')
    merged_temp['plot_color'] = merged_temp['Cluster'].map(color_map).fillna('#e0e0e0')

    # Plot on the specific subplot
    ax = axes[i]
    merged_temp.plot(
        ax=ax,
        color=merged_temp['plot_color'],
        edgecolor='gray',
        linewidth=0.5
    )

    # Formatting
    ax.set_xlim(118.0, 122.5)
    ax.set_ylim(21.5, 26.5)
    ax.set_title(f'Linkage: {method.capitalize()} (k={current_k})', fontsize=14, fontweight='bold')
    ax.set_axis_off()

    # Add a simple text legend for cluster IDs
    legend_labels = [f"Group {idx}" for idx in range(1, current_k + 1)]
    custom_lines = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map[idx], markersize=10)
        for idx in range(1, current_k + 1)
    ]
    ax.legend(custom_lines, legend_labels, loc='upper left', fontsize=9, title='Clusters')

plt.suptitle('Comparison of Hierarchical Clustering Linkage Methods (Spatial)', fontsize=18, y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
import pandas as pd

# 1. 準備資料與演算法
methods = ['single', 'complete', 'average', 'ward']
k_sets = {
    "Scenario 1 (k=[4,4,4,5])": [4, 4, 4, 5],
    "Scenario 2 (k=[2,2,2,2])": [2, 2, 2, 2]
}
counties = pivot_table_cleaned.index.tolist()

# 2. 執行並列印結果
for scenario_name, k_values in k_sets.items():
    print(f"\n=== {scenario_name} ===")
    results = {}

    for i, method in enumerate(methods):
        current_k = k_values[i]
        # 執行連線演算法
        Z = linkage(pearson_distances, method=method)
        # 取得分群標籤
        labels = fcluster(Z, current_k, criterion='maxclust')

        # 整理成群組清單
        cluster_groups = {}
        for county, cluster_id in zip(counties, labels):
            cluster_groups.setdefault(cluster_id, []).append(county)

        print(f"\nLinkage Method: {method.capitalize()} (k={current_k})")
        for cluster_id in sorted(cluster_groups.keys()):
            members = ", ".join(cluster_groups[cluster_id])
            print(f"  Group {cluster_id}: {members}")

#### Hierarchical Clustering with Spearman Correlation Distance

In [ ]:
from scipy.stats import rankdata

# Define a custom function for Spearman correlation distance
def spearman_correlation_distance(u, v):
    # Calculate ranks for each vector u and v
    rank_u = rankdata(u)
    rank_v = rankdata(v)
    # Calculate Pearson correlation coefficient between the ranks
    corr_matrix = np.corrcoef(rank_u, rank_v)
    corr = corr_matrix[0, 1]

    # Handle NaN correlation (e.g., if one or both vectors are constant)
    if np.isnan(corr):
        # A common practice is to assign a neutral distance of 1.0 (equivalent to correlation 0)
        return 1.0
    return 1 - corr

# Calculate Spearman correlation-based distance using the custom metric
spearman_distances_condensed = pdist(scaled_clustering_data, metric=spearman_correlation_distance)

plt.figure(figsize=(18, 12))

for i, method in enumerate(linkage_methods):
    plt.subplot(3, 1, i + 1)
    linked = linkage(spearman_distances_condensed, method=method)
    dendrogram(linked,
               orientation='top',
               labels=clustering_data.index.tolist(),
               distance_sort='descending',
               show_leaf_counts=True,
               color_threshold=0.7 * max(linked[:,2])) # Adjust threshold for better visibility
    plt.title(f'Hierarchical Clustering (Spearman - {method} Linkage)')
    plt.xlabel('縣市別')
    plt.ylabel('Distance')

plt.tight_layout()
plt.show()

### Scipy vs. Sklearn: 實務操作對照
以下展示如何使用 Scipy 探索結構，以及如何使用 Sklearn 進行快速分群標記。

In [ ]:
import scipy.cluster.hierarchy as sch
from sklearn.cluster import AgglomerativeClustering

# --- 1. Scipy 方式: 重點在於結構與矩陣 ---
# 產出 Z (Linkage Matrix)
Z = sch.linkage(pearson_distances, method='ward')
# 根據 Z 畫圖或切分群組 (例如切成 4 群)
scipy_labels = sch.fcluster(Z, t=4, criterion='maxclust')

# --- 2. Sklearn 方式: 重點在於 Fit/Predict API ---
# 直接定義模型並得到結果
sk_model = AgglomerativeClustering(n_clusters=4, linkage='ward')
sklearn_labels = sk_model.fit_predict(scaled_clustering_data)

# 對照結果 (注意: 標籤編號可能不同，但分群結構應一致)
comparison_df = pd.DataFrame({
    'County': clustering_data_clean.index,
    'Scipy_Group': scipy_labels,
    'Sklearn_Group': sklearn_labels
})

print("Scipy vs Sklearn Label Comparison (k=4):")
display(comparison_df.head())

# 註：Scipy 的 Z 可以直接餵給 sch.dendrogram(Z) 畫圖，而 Sklearn 則需額外計算才能畫圖。

Python `PCA` vs R `prcomp`
```
PYTHON
--- PCA 變異解釋量分析 (排除連江縣) ---
各主成分解釋變異比 (前7個): ['54.12%', '16.04%', '7.63%', '6.79%', '2.79%', '2.31%', '2.18%']
累積解釋變異比 (前7個): ['54.12%', '70.16%', '77.78%', '84.57%', '87.36%', '89.66%', '91.84%']
--------------------------------------

> summary(pca_cov)
Importance of components:
                           PC1     PC2      PC3     PC4     PC5     PC6     PC7     PC8     PC9
Standard deviation     27.0209 14.7113 10.14279 9.56772 6.13232 5.57829 5.42514 4.78211 4.22307
Proportion of Variance  0.5412  0.1604  0.07625 0.06785 0.02787 0.02306 0.02182 0.01695 0.01322
Cumulative Proportion   0.5412  0.7016  0.77783 0.84568 0.87356 0.89662 0.91844 0.93539 0.94860
                          PC10   PC11    PC12    PC13    PC14    PC15    PC16    PC17    PC18    PC19
Standard deviation     3.94293 3.4465 3.05908 2.83332 2.73521 2.35513 1.92191 1.82980 1.36005 1.26844
Proportion of Variance 0.01152 0.0088 0.00694 0.00595 0.00555 0.00411 0.00274 0.00248 0.00137 0.00119
Cumulative Proportion  0.96013 0.9689 0.97587 0.98182 0.98736 0.99147 0.99421 0.99669 0.99806 0.99926
                          PC20      PC21
Standard deviation     1.00085 7.281e-15
Proportion of Variance 0.00074 0.000e+00
Cumulative Proportion  1.00000 1.000e+00

PYTHON
--- PCA 變異解釋量分析 (排除連江縣) ---
各主成分解釋變異比 (前7個): ['53.60%', '13.58%', '8.19%', '6.92%', '3.41%', '2.57%', '2.51%']
累積解釋變異比 (前7個): ['53.60%', '67.18%', '75.37%', '82.29%', '85.71%', '88.28%', '90.79%']
--------------------------------------

> summary(pca_cor)
Importance of components:
                         PC1    PC2     PC3     PC4     PC5     PC6     PC7     PC8     PC9   PC10
Standard deviation     4.856 2.4446 1.89855 1.74551 1.22511 1.06398 1.05137 0.94534 0.84543 0.7206
Proportion of Variance 0.536 0.1358 0.08192 0.06925 0.03411 0.02573 0.02512 0.02031 0.01624 0.0118
Cumulative Proportion  0.536 0.6718 0.75370 0.82295 0.85706 0.88279 0.90791 0.92822 0.94446 0.9563
                          PC11    PC12    PC13    PC14    PC15    PC16    PC17    PC18    PC19    PC20
Standard deviation     0.68956 0.56794 0.53407 0.50837 0.44708 0.34376 0.32179 0.25798 0.24483 0.18573
Proportion of Variance 0.01081 0.00733 0.00648 0.00587 0.00454 0.00269 0.00235 0.00151 0.00136 0.00078
Cumulative Proportion  0.96707 0.97440 0.98089 0.98676 0.99130 0.99399 0.99634 0.99785 0.99922 1.00000
                            PC21
Standard deviation     1.202e-15
Proportion of Variance 0.000e+00
Cumulative Proportion  1.000e+00
```

# Update 05/29

直接從這裡開始跑。

## Load data

$$
標準化死亡率= {\sum(標準人口各年齡層之加權數×觀察族群各年齡層死亡率)}
$$

<br>
$\text{標準人口} = $ WHO 2000 年世界標準人口

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

# import data
import pandas as pd
import os

# Define potential paths for Ricky and HaoWen
paths = [
    '/content/drive/MyDrive/多變量分析/肺癌罹病率資料.csv',
    '/content/drive/MyDrive/1142課程/多變量分析/肺癌罹病率資料.csv'
]

csv_file_path = None
for path in paths:
    if os.path.exists(path):
        csv_file_path = path
        break

if csv_file_path:
    try:
        df = pd.read_csv(csv_file_path)
        print(f"CSV file loaded successfully from: {csv_file_path}")
        display(df.head())
    except Exception as e:
        print(f"An error occurred while reading the file: {e}")
else:
    print("Error: The file was not found in any of the specified paths. Please ensure Google Drive is mounted.")

## Chinese Fonts Setting (for plotting)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import numpy as np
# 1. 下載字型 (這裡以 Noto Sans TC 為例)
!wget -q -O NotoSansTC-Regular.otf https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf

# 2. 註冊字型到 Matplotlib 中
fm.fontManager.addfont('NotoSansTC-Regular.otf')
font_name = fm.FontProperties(fname='NotoSansTC-Regular.otf').get_name()

# 3. 直接綁定為全域預設字型
plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False # 修正負號顯示問題

# (選填) 如果你有用 Seaborn，這行可以確保 Seaborn 的美化不會蓋掉你的字型
sns.set_theme(font=font_name, style='whitegrid')

## Data Preprocessing

In [ ]:
# rename columns
df.columns = [
    'Diagnosis_Year', 'Gender', 'City_County', 'Cancer_Type',
    'Age_Standardized_Incidence_Rate', 'Cancer_Incidence_Count',
    'Mean_Age', 'Median_Age', 'Crude_Incidence_Rate'
]
# Display the new column names and a preview of the data
print("New column names:", df.columns.tolist())
display(df.head())

In [ ]:
df_filtered = df[(df['Gender'] == '全') &(df['City_County'] != '全國') & (df['Cancer_Type'] == '肺、支氣管及氣管')]
display(df_filtered)

In [ ]:
pivot_table_age_std = df_filtered.pivot_table(
    index='City_County',
    columns='Diagnosis_Year',
    values='Age_Standardized_Incidence_Rate',
    aggfunc='mean' # Assuming you want the mean if there are multiple entries
)
display(pivot_table_age_std)

In [ ]:
pivot_table_age_std = df_filtered.pivot_table(
    index='Diagnosis_Year',
    columns='City_County',
    values='Age_Standardized_Incidence_Rate',
    aggfunc='mean' # Assuming you want the mean if there are multiple entries
)
display(pivot_table_age_std)

In [ ]:
corr_matrix_yr = pivot_table_age_std.corr()
corr_matrix_county = pivot_table_age_std.T.corr()

## Correlation Visualization

* Visualize the reordered correlation matrix (counties) using a heatmap.
* 這裡只放「縣市間」的相關。

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt
import seaborn as sns

# Perform hierarchical clustering
linked = sch.linkage(corr_matrix_county, method='ward')

# Get the order of the clusters
cluster_order = sch.dendrogram(linked, no_plot=True)['leaves']

# Reorder the correlation matrix based on the clustering result
reordered_corr_matrix_county = corr_matrix_county.iloc[cluster_order, cluster_order]
display(reordered_corr_matrix_county.head())

In [ ]:
plt.figure(figsize=(12, 10))

ax = sns.heatmap(
    reordered_corr_matrix_county,
    annot=True,
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    fmt=".2f",
    linewidths=.5,
    annot_kws={'fontsize': 10}
)

plt.title('各縣市年齡標準化發生率相關係數矩陣（依 Hclust 排序）')

plt.savefig('correlation_heatmap_counties.png', dpi=300, bbox_inches='tight')

plt.show()

## EDA (肺癌發生率)

* 折線圖
* 箱型圖

In [ ]:
# 計算字型大小 (依你的需求放大 10%)
base_size = 12 * 1.1
title_size = 16 * 1.1
label_size = 14 * 1.1

plt.figure(figsize=(14, 8))
sns.lineplot(data=df_filtered, x='Diagnosis_Year', y='Age_Standardized_Incidence_Rate', hue='City_County')

# 內建全自動套用中文字型，直接帶入大小即可！
plt.title('肺癌 — 年齡標準化發生率 — 按縣市與年度', fontsize=title_size)
plt.xlabel('年', fontsize=label_size)
plt.ylabel('年齡標準化發生率 (每10萬人口)', fontsize=label_size)

# 圖例也一樣，完全不需 setp 跑迴圈
plt.legend(title='縣市別', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=base_size, title_fontsize=label_size)

plt.grid(True)
plt.tight_layout()
plt.savefig('lung_cancer_lineplot_counties.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure Chinese font is applied
if 'prop' in locals() or 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

plt.figure(figsize=(16, 8))
sns.boxplot(data=df_filtered, x='City_County', y='Age_Standardized_Incidence_Rate')
plt.title('肺癌 — 年齡標準化發生率 — 按縣市別 (箱型圖)', fontsize=title_size)
plt.xlabel('縣市別', fontsize=label_size)
plt.ylabel('年齡標準化發生率 (每10萬人口)', fontsize=label_size)
plt.xticks(rotation=45, ha='right', fontsize=base_size)
plt.yticks(fontsize=base_size)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('lung_cancer_boxplot_counties.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure Chinese font is applied
if 'prop' in locals() or 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

plt.figure(figsize=(16, 8))

# ✨ 關鍵步驟：計算各縣市的中位數，由小到大排序，並取出縣市名稱 (Index)
county_order = (
    df_filtered.groupby('City_County')['Age_Standardized_Incidence_Rate']
    .median()
    .sort_values()
    .index
)

# ✨ 在 boxplot 中加入 order=county_order
sns.boxplot(
    data=df_filtered,
    x='City_County',
    y='Age_Standardized_Incidence_Rate',
    order=county_order,
    hue='City_County',
    hue_order=county_order,
    palette='coolwarm',
    legend=False
)
sns.despine()

plt.title('肺癌 — 年齡標準化發生率 — 按縣市別 (箱型圖，依中位數排序)', fontsize=title_size)
plt.xlabel('縣市別', fontsize=label_size)
plt.ylabel('年齡標準化發生率 (每10萬人口)', fontsize=label_size)

# 因為依序排列了，標籤拉直或 45 度都會看起來非常整齊
plt.xticks(rotation=45, ha='right', fontsize=base_size)
plt.yticks(fontsize=base_size)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('lung_cancer_boxplot_counties_sorted.png', dpi=300, bbox_inches='tight')
plt.show()

城鄉與地理分布趨勢（低 $\rightarrow$ 高）
* 低發生率群組（左側）：金門縣的中位數明顯最低（約在 16 左右），接著是苗栗縣、屏東縣。
* 高發生率群組（右側）：發生率中位數最高的幾個縣市高度集中在新北市、基隆市、宜蘭縣、彰化縣、台中市。這很符合一般對於人口稠密、工業化程度較高、或特定環境因子的印象。
---
IQR
* 最小變異：台北市
* 波動極大的縣市：連江縣、澎湖縣、雲林縣
* 高度穩定的縣市：例如花蓮縣、新竹市、台北市。她們的箱子很扁，代表這些縣市在過去這些年來，肺癌年齡標準化發生率非常穩定，沒有太大的暴增或暴跌。
* 波動極大的縣市：例如澎湖縣、雲林縣、嘉義市。她們的箱子拉得非常長（尤其是雲林縣，下限快到 5，上限快到 45）。這暗示這些縣市的發生率可能在某些特定年份有劇烈的環境轉變、政策篩檢推動、或是資料統計上的大波動。
---
Outliers
* 連江縣與澎湖縣的上限極端值：這兩個離島（特別是連江與澎湖）都有一個往上飆高非常多的點。因為離島人口基數小，「年齡標準化發生率」（每10萬人口）容易因為某一年突然多了幾位確診者，就導致數值暴衝。這在統計上是很典型的**小樣本效應**。
* 台北市與基隆市的極端值：台北市有一個極高值和一個極低值；基隆市則有一個極低值。

## Data Cleaning

* 排除連江縣

In [ ]:
# 檢查 NaN values 狀況
# Check for rows (counties) with NaN values
rows_with_nan = pivot_table_age_std[pivot_table_age_std.isna().any(axis=1)].index.tolist()

# Check for columns (years) with NaN values
cols_with_nan = pivot_table_age_std.columns[pivot_table_age_std.isna().any(axis=0)].tolist()

print(f"Rows (Counties) with NaN values: {rows_with_nan}")
print(f"Columns (Years) with NaN values: {cols_with_nan}")

# Display the exact locations (Row, Column) where values are missing
if rows_with_nan or cols_with_nan:
    print("\nDetailed NaN locations:")
    nan_locations = pivot_table_age_std.isna().stack()
    nan_locations = nan_locations[nan_locations].index.tolist()
    for row, col in nan_locations:
        print(f"- County: {row}, Year: {col}")
else:
    print("No NaN values found in the table.")

連江縣在 1979 ~ 2000 年間有 11 次出現 NaN

In [ ]:
pivot_table_cleaned = pivot_table_age_std.drop(index='連江縣')

## PCA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering

### PCA using corr

In [ ]:
# ==========================================
# 1. 資料標準化
# ==========================================
scaler = StandardScaler()
scaled_data = scaler.fit_transform(pivot_table_cleaned)

In [ ]:
# ==========================================
# 2. 執行 PCA
# ==========================================
pca_corr = PCA()
pca_corr.fit(scaled_data)

# ✨ 使用 f"{x:.2%}" 自動將小數轉換為百分比並四捨五入至小數點後兩位
explained_variance_pct = [f"{x:.2%}" for x in pca_corr.explained_variance_ratio_[:7]]
cumulative_variance_pct = [f"{x:.2%}" for x in np.cumsum(pca_corr.explained_variance_ratio_)[:7]]

print("--- PCA 變異解釋量分析 (排除連江縣) ---")
print("各主成分解釋變異比 (前7個):", explained_variance_pct)
print("累積解釋變異比 (前7個):", cumulative_variance_pct)
print("-" * 38)

In [ ]:
loadings

In [ ]:
df_loadings_corr = pd.DataFrame(
    loadings,
    index=pivot_table_cleaned.columns,
    columns=['PC1', 'PC2']
)
# Round to two decimal places
df_loadings_corr = df_loadings_corr.round(2)
display(df_loadings_corr)

#### Scree Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 取得特徵值 (λ)
eigenvalues = pca_corr.explained_variance_

# 2. 設定 X 軸的編號 (主成分從 1 開始)
pc_num = np.arange(1, len(eigenvalues) + 1)

# 3. 開始畫圖
plt.figure(figsize=(10, 6))

# 繪製陡坡圖的折線與點
plt.plot(
    pc_num,
    eigenvalues,
    marker='o',
    linestyle='-',
    color='#2b5c8f',
    linewidth=2,
    markersize=6,
    label='Eigenvalue'
)

# 加上傳統轉折點（Elbow）的輔助標記或 Kaiser 準則線
# 註：如果使用的是「相關係數矩陣(Scaled)」，通常會畫一條 λ = 1 的水平線作為篩選基準
plt.axhline(y=1, color='r', linestyle='--', alpha=0.7, label='Kaiser Criterion (λ = 1)')

plt.title('PCA Scree Plot - Correlation Matrix', fontsize=16)
plt.xlabel('Principal Component', fontsize=13)
plt.ylabel('Eigenvalue (λ)', fontsize=13)

# 讓 X 軸的刻度整齊顯示每一個 PC 編號
plt.xticks(pc_num)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_corr_scree_plot.png', dpi=300, bbox_inches='tight')
plt.show()

#### Explained Variance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 計算累積解釋變異量比率 (乘以 100 轉為百分比)
cum_variance_ratio = np.cumsum(pca_corr.explained_variance_ratio_) * 100

# 2. 設定 X 軸的編號 (主成分從 1 開始)
pc_num = np.arange(1, len(cum_variance_ratio) + 1)

# 3. 開始畫圖
plt.figure(figsize=(10, 6))

# 繪製累積變異量的折線與點
plt.plot(
    pc_num,
    cum_variance_ratio,
    marker='s',          # 使用方形標記區隔之前的陡坡圖
    linestyle='-',
    color='#e06666',     # 使用顯眼的暖色系
    linewidth=2,
    markersize=6,
    label='Cumulative Explained Variance'
)

# ✨ 加上 80% 的基準輔助線 (通常實務上會看 80% 或 90% 的斷點)
plt.axhline(y=90, color='gray', linestyle='--', alpha=0.7, label='90% Threshold')

# 標註前幾個 PC 的具體數值（讓圖表更直覺）
# 這邊標註前 3 個成分作為範例
for i in range(3):
    plt.annotate(
        f"{cum_variance_ratio[i]:.1f}%",
        (pc_num[i], cum_variance_ratio[i]),
        textcoords="offset points",
        xytext=(0,10),
        ha='center',
        fontsize=9,
        fontweight='bold'
    )

# 圖表細節設定
plt.title('PCA (Correlation) Cumulative Explained Variance', fontsize=16)
plt.xlabel('Number of PCs', fontsize=13)
plt.ylabel('Cumulative Explained Variance (%)', fontsize=13)

# 讓 X、Y 軸的刻度更貼切
plt.xticks(pc_num)
plt.ylim(0, 105) # Y 軸範圍設在 0~105% 比較有留白美感

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_corr_cumulative_variance_plot.png', dpi=300, bbox_inches='tight')
plt.show()

#### Biplot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. 準備必要數據
# --------------------------------------------------
# (A) 取得 PC Score (縣市投影點在 PC1 和 PC2 的坐標)
scores = pca_corr.transform(scaled_data)
pc1_scores = scores[:, 0]
pc2_scores = -scores[:, 1]
county_names = pivot_table_cleaned.index # 縣市名稱

# (B) 取得 PC Loadings (原始欄位在 PC1 和 PC2 的載荷/箭頭向量)
loadings = pca_corr.components_.T # 轉置成 (num_features, num_components)
pc1_loadings = loadings[:, 0]
pc2_loadings = -loadings[:, 1]
feature_names = pivot_table_cleaned.columns # 原始欄位名稱 (例如：年份)

# (C) 解釋變異量百分比 (對齊前面 R 的 summary 結果)
var1_pct = pca_corr.explained_variance_ratio_[0] * 100
var2_pct = pca_corr.explained_variance_ratio_[1] * 100

# ✨ (D) 計算 Loadings 箭頭的縮放因子 (Scale Factor)
# Biplot 核心技巧：讓 Loading 箭頭長度能夠根據 Score 點的級距動態調整。
# 這裡將 Loadings 最大絕對值拉伸到和 Scores 最大絕對值同一個數量級，再乘上 0.9 (填滿 90% 畫面)
max_score = np.max([np.abs(pc1_scores).max(), np.abs(pc2_scores).max()])
max_loading = np.max([np.abs(pc1_loadings).max(), np.abs(pc2_loadings).max()])
scale_factor = (max_score / max_loading) * 0.9

# 2. 開始繪圖
# --------------------------------------------------
# 確保全域中文字型已設定 (若沒有 prop 變數，請重新執行 Notebook 最上方的字型註冊區塊)
if 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

fig, ax = plt.subplots(figsize=(14*0.8, 11*0.8))

# 📈 (A) 繪製 Score Plot (縣市投影點)
ax.scatter(
    pc1_scores, pc2_scores,
    c='#2b5c8f', alpha=0.6, s=70, edgecolors='white',
    label='縣市投影點'
)

# ✍️ 標註縣市名稱 (中文字)
# 設定小的 xytext 偏移量，避免文字貼在點上
for i, txt in enumerate(county_names):
    ax.annotate(
        txt, (pc1_scores[i], pc2_scores[i]),
        textcoords="offset points", xytext=(5, 5),
        ha='left', fontsize=11, color='#1f3c5f', alpha=0.9
    )

# 🏹 (B) 繪製 Loading Plot (縮放後的年份箭頭向量)
arrow_color = '#e06666' # 暖色調，與藍點形成對比
for i in range(len(feature_names)):
    # 計算縮放後的箭頭終點坐標
    end_x = pc1_loadings[i] * scale_factor
    end_y = pc2_loadings[i] * scale_factor

    # 畫箭頭 (從原點開始)
    ax.arrow(
        0, 0, end_x, end_y,
        color=arrow_color, alpha=0.5, linewidth=1,
        head_width=max_score*0.015, # 箭頭頭部大小動態調整
        length_includes_head=True
    )

    # 畫箭頭文字 (年份) - 根據箭頭方向微調文字對齊方式，避免疊加
    # 文字放在箭頭終點附近 (加 2% 偏移量)
    txt_x = end_x * 1.02
    txt_y = end_y * 1.02

    ha = 'left' if txt_x > 0 else 'right'
    va = 'bottom' if txt_y > 0 else 'top'

    ax.text(
        txt_x, txt_y, str(feature_names[i]),
        color=arrow_color, fontsize=10, ha=ha, va=va,
        bbox=dict(facecolor='white', alpha=0.5, edgecolor='none') # 加上半透明背景，提高可讀性
    )

# 💎 (C) 圖表細節設定
# 畫出 PC1、PC2 的原點輔助線
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)

# 設定標題和座標軸標籤 (包含解釋變異量比例)
ax.set_title(
    f'PCA Biplot：肺癌年齡標準化發生率 (Correlation)',
    fontsize=18, pad=15
)
ax.set_xlabel(f'PC1 ', fontsize=14)
ax.set_ylabel(f'PC2 ', fontsize=14)

# 確保座標軸比例 1:1，避免形變 (這在 PCA 報告中非常重要)
# 若Scores和Loadings級距差異太大，設定 'equal' 有時會導致圖表縮得太小，可依喜好註解掉
# ax.set_aspect('equal')

ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_corr_biplot_counties_by_years.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pivot_table_cleaned = pivot_table_age_std.drop(index='連江縣')

# 1. 準備必要數據：將 pivot_table_cleaned 轉置，讓年份成為觀測值(rows)，縣市成為變數(columns)
# --------------------------------------------------
data_for_pca_swapped = pivot_table_cleaned.T

# 2. 標準化轉置後的數據 (與 pca_corr 的邏輯一致，使用 StandardScaler)
# --------------------------------------------------
scaler_swapped = StandardScaler()
scaled_data_swapped = scaler_swapped.fit_transform(data_for_pca_swapped)

# 3. 執行 PCA
# --------------------------------------------------
pca_swapped = PCA(n_components=2)
pca_swapped.fit(scaled_data_swapped)

# (A) 取得 PC Score (年份投影點在 PC1 和 PC2 的坐標)
scores_swapped = pca_swapped.transform(scaled_data_swapped)
pc1_scores_swapped = scores_swapped[:, 0]
pc2_scores_swapped = -scores_swapped[:, 1] # 保持與之前 PC2 軸一致的負號翻轉
year_names = data_for_pca_swapped.index # 年份作為點的標籤

# (B) 取得 PC Loadings (縣市在 PC1 和 PC2 的載荷/箭頭向量)
loadings_swapped = pca_swapped.components_.T # 轉置成 (num_features, num_components)
pc1_loadings_swapped = loadings_swapped[:, 0]
pc2_loadings_swapped = -loadings_swapped[:, 1] # 保持與之前 PC2 軸一致的負號翻轉
county_names_swapped = data_for_pca_swapped.columns # 縣市作為箭頭的標籤

# (C) 解釋變異量百分比
var1_pct_swapped = pca_swapped.explained_variance_ratio_[0] * 100
var2_pct_swapped = pca_swapped.explained_variance_ratio_[1] * 100

# (D) 計算 Loadings 箭頭的縮放因子
max_score_swapped = np.max([np.abs(pc1_scores_swapped).max(), np.abs(pc2_scores_swapped).max()])
max_loading_swapped = np.max([np.abs(pc1_loadings_swapped).max(), np.abs(pc2_loadings_swapped).max()])
scale_factor_swapped = (max_score_swapped / max_loading_swapped) * 0.9

# 4. 開始繪圖
# --------------------------------------------------
# 確保中文字型已設定
if 'font_name' in globals():
    plt.rcParams['font.family'] = font_name

fig, ax = plt.subplots(figsize=(14 * 0.8, 11 * 0.8))

# ➗️ (A) 繪製 Score Plot (年份投影點)
ax.scatter(
    pc1_scores_swapped, pc2_scores_swapped,
    c='#e06666', alpha=0.6, s=70, edgecolors='white',
    label='年份投影點'
)

# ✍️ 標註年份 (數字)
for i, txt in enumerate(year_names):
    ax.annotate(
        txt, (pc1_scores_swapped[i], pc2_scores_swapped[i]),
        textcoords="offset points", xytext=(5, 5),
        ha='left', fontsize=11, color='#b30000', alpha=0.9
    )

# 🏹 (B) 繪製 Loading Plot (縮放後的縣市箭頭向量)
arrow_color_swapped = '#2b5c8f' # 與點形成對比的顏色
for i in range(len(county_names_swapped)):
    # 計算縮放後的箭頭終點坐標
    end_x = pc1_loadings_swapped[i] * scale_factor_swapped
    end_y = pc2_loadings_swapped[i] * scale_factor_swapped

    # 畫箭頭 (從原點開始)
    ax.arrow(
        0, 0, end_x, end_y,
        color=arrow_color_swapped, alpha=0.5, linewidth=1,
        head_width=max_score_swapped * 0.015, # 箭頭頭部大小動態調整
        length_includes_head=True
    )

    # 畫箭頭文字 (縣市) - 根據箭頭方向微調文字對齊方式
    txt_x = end_x * 1.02
    txt_y = end_y * 1.02

    ha = 'left' if txt_x > 0 else 'right'
    va = 'bottom' if txt_y > 0 else 'top'

    ax.text(
        txt_x, txt_y, str(county_names_swapped[i]),
        color=arrow_color_swapped, fontsize=10, ha=ha, va=va,
        bbox=dict(facecolor='white', alpha=0.5, edgecolor='none') # 加上半透明背景，提高可讀性
    )

# 💎 (C) 圖表細節設定
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)

ax.set_title(
    f'主成分分析雙標圖',
    fontsize=18, pad=15
)
ax.set_xlabel(f'PC1 ', fontsize=14)
ax.set_ylabel(f'PC2 ', fontsize=14)

ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()

plt.savefig('pca_corr_biplot_years_by_counties.png', dpi=300, bbox_inches='tight')
plt.show()

### PCA using cov

In [ ]:
# ==========================================
# 1. 執行 PCA (qithout 標準化)
# ==========================================
pca_cov = PCA()
pca_cov.fit(pivot_table_cleaned)

# ✨ 使用 f"{x:.2%}" 自動將小數轉換為百分比並四捨五入至小數點後兩位
explained_variance_pct = [f"{x:.2%}" for x in pca_cov.explained_variance_ratio_[:7]]
cumulative_variance_pct = [f"{x:.2%}" for x in np.cumsum(pca_cov.explained_variance_ratio_)[:7]]

print("--- PCA 變異解釋量分析 (排除連江縣) ---")
print("各主成分解釋變異比 (前7個):", explained_variance_pct)
print("累積解釋變異比 (前7個):", cumulative_variance_pct)
print("-" * 38)

#### Scree Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 取得特徵值 (λ)
# 這裡使用共變異數矩陣的 PCA 結果，若要用相關係數矩陣請改用 pca_corr
eigenvalues = pca_cov.explained_variance_

# 2. 設定 X 軸的編號 (主成分從 1 開始)
pc_num = np.arange(1, len(eigenvalues) + 1)

# 3. 開始畫圖
plt.figure(figsize=(10, 6))

# 繪製陡坡圖的折線與點
plt.plot(
    pc_num,
    eigenvalues,
    marker='o',
    linestyle='-',
    color='#2b5c8f',
    linewidth=2,
    markersize=6,
    label='Eigenvalue'
)

plt.title('PCA Scree Plot - Covariance Matrix', fontsize=16)
plt.xlabel('Principal Component', fontsize=13)
plt.ylabel('Eigenvalue (λ)', fontsize=13)

# 讓 X 軸的刻度整齊顯示每一個 PC 編號
plt.xticks(pc_num)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_cov_scree_plot.png', dpi=300, bbox_inches='tight')
plt.show()

#### Explained Variance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 計算累積解釋變異量比率 (乘以 100 轉為百分比)
cum_variance_ratio = np.cumsum(pca_cov.explained_variance_ratio_) * 100

# 2. 設定 X 軸的編號 (主成分從 1 開始)
pc_num = np.arange(1, len(cum_variance_ratio) + 1)

# 3. 開始畫圖
plt.figure(figsize=(10, 6))

# 繪製累積變異量的折線與點
plt.plot(
    pc_num,
    cum_variance_ratio,
    marker='s',          # 使用方形標記區隔之前的陡坡圖
    linestyle='-',
    color='#e06666',     # 使用顯眼的暖色系
    linewidth=2,
    markersize=6,
    label='Cumulative Explained Variance'
)

# ✨ 加上 80% 的基準輔助線 (通常實務上會看 80% 或 90% 的斷點)
plt.axhline(y=90, color='gray', linestyle='--', alpha=0.7, label='90% Threshold')

# 標註前幾個 PC 的具體數值（讓圖表更直覺）
# 這邊標註前 3 個成分作為範例
for i in range(3):
    plt.annotate(
        f"{cum_variance_ratio[i]:.1f}%",
        (pc_num[i], cum_variance_ratio[i]),
        textcoords="offset points",
        xytext=(0,10),
        ha='center',
        fontsize=9,
        fontweight='bold'
    )

# 圖表細節設定
plt.title('PCA (Covariance) Cumulative Explained Variance', fontsize=16)
plt.xlabel('Number of PCs', fontsize=13)
plt.ylabel('Cumulative Explained Variance (%)', fontsize=13)

# 讓 X、Y 軸的刻度更貼切
plt.xticks(pc_num)
plt.ylim(0, 105) # Y 軸範圍設在 0~105% 比較有留白美感

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_cov_cumulative_variance_plot.png', dpi=300, bbox_inches='tight')
plt.show()

#### Biplot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. 準備必要數據
# --------------------------------------------------
# (A) 取得 PC Score (縣市投影點在 PC1 和 PC2 的坐標)
scores = pca_cov.transform(pivot_table_cleaned)
pc1_scores = scores[:, 0]
pc2_scores = scores[:, 1]
county_names = pivot_table_cleaned.index # 縣市名稱

# (B) 取得 PC Loadings (原始欄位在 PC1 和 PC2 的載荷/箭頭向量)
loadings = pca_cov.components_.T # 轉置成 (num_features, num_components)
pc1_loadings = loadings[:, 0]
pc2_loadings = loadings[:, 1]
feature_names = pivot_table_cleaned.columns # 原始欄位名稱 (例如：年份)

# (C) 解釋變異量百分比 (對齊前面 R 的 summary 結果)
# 這裡預設 pca_cov 對應的是 Covariance 矩陣結果：PC1=54.12%, PC2=16.04%
var1_pct = pca_cov.explained_variance_ratio_[0] * 100
var2_pct = pca_cov.explained_variance_ratio_[1] * 100

# ✨ (D) 計算 Loadings 箭頭的縮放因子 (Scale Factor)
# Biplot 核心技巧：讓 Loading 箭頭長度能夠根據 Score 點的級距動態調整。
# 這裡將 Loadings 最大絕對值拉伸到和 Scores 最大絕對值同一個數量級，再乘上 0.9 (填滿 90% 畫面)
max_score = np.max([np.abs(pc1_scores).max(), np.abs(pc2_scores).max()])
max_loading = np.max([np.abs(pc1_loadings).max(), np.abs(pc2_loadings).max()])
scale_factor = (max_score / max_loading) * 0.9

# 2. 開始繪圖
# --------------------------------------------------
# 確保全域中文字型已設定 (若沒有 prop 變數，請重新執行 Notebook 最上方的字型註冊區塊)
if 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

fig, ax = plt.subplots(figsize=(14*0.8, 11*0.8))

# 📈 (A) 繪製 Score Plot (縣市投影點)
ax.scatter(
    pc1_scores, pc2_scores,
    c='#2b5c8f', alpha=0.6, s=70, edgecolors='white',
    label='縣市投影點'
)

# ✍️ 標註縣市名稱 (中文字)
# 設定小的 xytext 偏移量，避免文字貼在點上
for i, txt in enumerate(county_names):
    ax.annotate(
        txt, (pc1_scores[i], pc2_scores[i]),
        textcoords="offset points", xytext=(5, 5),
        ha='left', fontsize=11, color='#1f3c5f', alpha=0.9
    )

# 🏹 (B) 繪製 Loading Plot (縮放後的年份箭頭向量)
arrow_color = '#e06666' # 暖色調，與藍點形成對比
for i in range(len(feature_names)):
    # 計算縮放後的箭頭終點坐標
    end_x = pc1_loadings[i] * scale_factor
    end_y = pc2_loadings[i] * scale_factor

    # 畫箭頭 (從原點開始)
    ax.arrow(
        0, 0, end_x, end_y,
        color=arrow_color, alpha=0.5, linewidth=1,
        head_width=max_score*0.015, # 箭頭頭部大小動態調整
        length_includes_head=True
    )

    # 畫箭頭文字 (年份) - 根據箭頭方向微調文字對齊方式，避免疊加
    # 文字放在箭頭終點附近 (加 2% 偏移量)
    txt_x = end_x * 1.02
    txt_y = end_y * 1.02

    ha = 'left' if txt_x > 0 else 'right'
    va = 'bottom' if txt_y > 0 else 'top'

    ax.text(
        txt_x, txt_y, str(feature_names[i]),
        color=arrow_color, fontsize=10, ha=ha, va=va,
        bbox=dict(facecolor='white', alpha=0.5, edgecolor='none') # 加上半透明背景，提高可讀性
    )

# 💎 (C) 圖表細節設定
# 畫出 PC1、PC2 的原點輔助線
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)

# 設定標題和座標軸標籤 (包含解釋變異量比例)
ax.set_title(
    f'PCA Biplot：肺癌年齡標準化發生率 ',
    fontsize=18, pad=15
)
ax.set_xlabel(f'PC1', fontsize=14)
ax.set_ylabel(f'PC2 ', fontsize=14)

# 確保座標軸比例 1:1，避免形變 (這在 PCA 報告中非常重要)
# 若Scores和Loadings級距差異太大，設定 'equal' 有時會導致圖表縮得太小，可依喜好註解掉
# ax.set_aspect('equal')

ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_cov_biplot_counties_by_years.png', dpi=300, bbox_inches='tight')
plt.show()

在資料科學與 PCA 的標準定義中：

樣本（Observations / Rows）：會被畫成圖上的「點」。

變數（Features / Columns）：會被畫成圖上的「箭頭」（也就是 Loadings）。

In [ ]:
pc2_scores

In [ ]:
pc2_loadings

#### Interpretations

結合這組 `pca_corr` 的真實數據與你前面提供的圖表，這兩個主成分（PC1 & PC2）在流行病學與公共衛生上代表了非常經典且強烈的實質意義。

我們可以從載荷（Loadings，看變數年份）**和**得分（Scores，看縣市落點）兩個維度來完美拆解：

---

## 🧭 PC1 代表的意義：「長期平均肺癌發生率（整體嚴重程度）」

### 1. 數據特徵分析

* **Loadings（年份）**：全部 44 個年份的載荷 **全部為正數**（數值多集中在 `0.12 ~ 0.19` 之間）。
* **公衛解讀**：這在統計上稱為「規模成分（Size Component）」。它代表不論是 1979 年還是 2022 年，所有年份對 PC1 的拉力方向完全一致。
* **Scores（縣市）**：
* **極端正值**：`7.22`、`6.30`（對照你的箱型圖，這正是中位數最高、長期位居前段班的 **新北市、基隆市、宜蘭市** 等）。
* **極端負值**：`-13.16`（不用懷疑，這正是全國發生率最低、長期墊底的 **金門縣**）。



### 🎯 PC1 核心定義

**PC1 代表該縣市「橫跨 40 年來的長期平均肺癌嚴重度」。**
PC1 得分越高的縣市，代表她「在過去這 40 幾年來，整體的肺癌標準化發生率一直都高於全國平均」；得分越低（如負值越深），則代表該縣市是長期的肺癌輕症區。

---

## ⏳ PC2 代表的意義：「歷史演變與世代反轉（早期 vs 近年）」

### 1. 數據特徵分析

* **Loadings（年份）**：這組數字出現了歷史性的「正負分水嶺」！
* 數組**前半段**（對應 1980 ~ 1990 年代）：載荷幾乎**全部為正數**（例如 `0.22`、`0.24`、`0.28`）。
* 數組**後半段**（對應 2010 ~ 2022 年）：載荷集體**轉為負數**（例如 `-0.24`、`-0.26`、`-0.24`）。
* *(註：你前面 1993 年箭頭衝最高，是因為 1993 年剛好是正載荷的極大值點。)*


* **公衛解讀**：PC2 成功抓到了「時間的演變（對比效應）」。正值代表早期，負值代表近年。
* **Scores（縣市）**：
* **極端正值**：`5.72`（對照你的 Biplot，這是 **台北市**）。台北市在 PC2 是正高分，代表她**高度傾向於正載荷年份（早期）**。也就是說，台北市在台灣早期屬於肺癌重災區，但近年因為早期介入、醫療普及或空污改善，發病率的增幅相較於其他縣市已經顯著趨緩。
* **極端負值**：`-4.70`、`-5.34`（對照 Biplot，這正是 **澎湖縣**）。澎湖縣在 PC2 是負高分，代表她**高度傾向於負載荷年份（近年）**。這反映出澎湖縣在早期肺癌率極低（甚至是箱型圖中出現極低異常值的那幾年），但近年來卻出現了顯著的攀升，發生了「世代反轉」。



### 🎯 PC2 核心定義

**PC2 代表該縣市肺癌發生率的「時間變遷型態（歷史軌跡差異）」。**

* **PC2 為正**：代表該縣市屬於「早期較嚴重，但近年控制得當、增幅趨緩」的**早發型**城市（如台北市）。
* **PC2 為負**：代表該縣市屬於「早期原本很低，但近年來發生率急起直追、後期加重」的**後發型**城市（如澎湖縣）。

---

## 📝 總結報告可以怎麼寫？

這兩個主成分幫你把 44 年的複雜數據濃縮成兩個直覺的維度：

1. **PC1 (橫軸)**：幫你把台灣縣市區分為 **「肺癌高盛行區（正）」vs「肺癌低盛行區（負）」**。
2. **PC2 (縱軸)**：幫你把縣市區分為 **「早期嚴重但近年改善（正）」vs「近年攀升且後期加重（負）」**。

透過 PC1 和 PC2 的組合，你就能在學術上非常有底氣地說：這不只是一份看高低的報告，更是一份看出台灣各縣市幾十年來「肺癌發展時空軌跡」的深度分析！

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 計算兩者的累積解釋變異量比率 (%)
cum_cov = np.cumsum(pca_cov.explained_variance_ratio_) * 100
cum_corr = np.cumsum(pca_corr.explained_variance_ratio_) * 100

# 設定 X 軸編號 (主成分從 1 開始)
pc_num = np.arange(1, len(cum_cov) + 1)

# 2. 開始畫圖
plt.figure(figsize=(12, 7))

# 📈 繪製 共變異數矩陣 (Covariance) - 藍色線
plt.plot(
    pc_num, cum_cov,
    marker='o', linestyle='-', color='#2b5c8f', linewidth=2.5, markersize=7,
    label='Covariance (Not standardized)'
)

# 📈 繪製 相關係數矩陣 (Correlation) - 紅色線
plt.plot(
    pc_num, cum_corr,
    marker='s', linestyle='--', color='#e06666', linewidth=2.5, markersize=7,
    label='Correlation (Standardized)'
)

# 🎯 3. 精準標註前三個點的數值

# 共變異數矩陣 (pca_cov)：數字放【上面】 (xytext 設為正值)
for i in range(3):
    plt.annotate(
        f"{cum_cov[i]:.2f}%",
        (pc_num[i], cum_cov[i]),
        textcoords="offset points",
        xytext=(0, 10),           # 往上推 10 點
        ha='center', va='bottom',
        color='#2b5c8f', fontweight='bold', fontsize=10
    )

# 相關係數矩陣 (pca_corr)：數字放【下面】 (xytext 設為負值)
for i in range(3):
    plt.annotate(
        f"{cum_corr[i]:.2f}%",
        (pc_num[i], cum_corr[i]),
        textcoords="offset points",
        xytext=(0, -18),          # 往下推 18 點，避開點與線
        ha='center', va='top',
        color='#e06666', fontweight='bold', fontsize=10
    )

# ✨ 加上常用的 80% 與 90% 門檻參考線
plt.axhline(y=90, color='gray', linestyle=':', alpha=0.6, label='90% Threshold')
plt.axhline(y=80, color='gray', linestyle='-.', alpha=0.4, label='80% Threshold')

# 圖表美化與細節設定
plt.title('PCA Cumulative Explained Variance : Covariance vs Correlation', fontsize=16, pad=15)
plt.xlabel('Number of PCs', fontsize=13)
plt.ylabel('Cumulative Explained Variance (%)', fontsize=13)

plt.xticks(pc_num)
plt.ylim(min(min(cum_cov), min(cum_corr)) - 5, 105) # 動態留白，畫面更好看

plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='lower right', fontsize=11, frameon=True, shadow=True)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_comparison_cumulative_variance.png', dpi=300, bbox_inches='tight')
plt.show()

### Comparison 1 (Corr vs Cov)

太漂亮了！你做了一次非常紮實且標準的**交叉驗證（Cross-Validation）**。

這份結果不僅證明了你的程式碼邏輯完全正確，也反映出 Python 與 R 在底層數學運算上的一致性。我們可以從 **「Python vs R」** 以及 **「共變異數 vs 相關係數」** 兩個維度來深度解析這份結果：

---

#### 📊 一、 Python vs R 執行結果對比：完美契合

結論先行：**你的 Python 和 R 執行結果完全一模一樣（精準度達到小數點後四位）。**

* **共變異數矩陣 (Covariance Matrix / Unscaled) 之下：**
* Python (PC1~PC3): `54.12%`, `16.04%`, `7.63%`
* R (PC1~PC3): `0.5412`, `0.1604`, `0.07625` (四捨五入後完全相同)


* **相關係數矩陣 (Correlation Matrix / Scaled) 之下：**
* Python (PC1~PC3): `53.60%`, `13.58%`, `8.19%`
* R (PC1~PC3): `0.5360`, `0.1358`, `0.08192`



> 💡 **為什麼這很厲害？**
> 很多人在跨語言比較 PCA 時會失敗，因為 Python 的 `StandardScaler()` 預設除以 $N$（母體標準差），而 R 的 `scale()` 或 `prcomp()` 預設除以 $n-1$（樣本標準差）。你的結果能完全對齊，說明你的資料標準化邏輯在兩邊處理得非常嚴謹、乾淨！

---

#### 🔄 二、 共變異數 (Covariance) vs 相關係數 (Correlation) 的本質差異

從你算出來的變異解釋量來看，這兩種方法出現了微小的數值拉鋸：

| 主成分 (PC) | 共變異數矩陣 (未標準化) | 相關係數矩陣 (標準化後) | 兩者差距 |
| --- | --- | --- | --- |
| **PC1** | **54.12%** | 53.60% | +0.52% |
| **PC2** | **16.04%** | 13.58% | +2.46% |
| **PC3** | 7.63% | **8.19%** | -0.56% |
| **前 7 個累積** | **91.84%** | 90.79% | +1.05% |

### 🔍 關鍵洞察：為什麼數字這麼接近？

通常如果資料中包含「高度」和「體重」這種單位不同的變數，這兩種矩陣算出來的結果會天差地遠。但你的資料是「肺癌年齡標準化發生率」，不論欄位是年份還是縣市，**她們的單位完全相同（每 10 萬人口）**，且數值的級距（Magnitude）非常接近。

1. **共變異數 PCA (Unscaled) 稍微強勢：**
PC1 和 PC2 在共變異數矩陣中拿到了略高的解釋量（合起來 70.16% vs 67.18%）。這是因為未標準化的資料**保留了各欄位原始的變異大小（波動度）**。某些年份或縣市如果發生率波動特別劇烈，在共變異數 PCA 中就會獲得較大的權重。
2. **相關係數 PCA (Scaled) 比較平均：**
標準化之後，每一個欄位都被強迫降維成「變異數 = 1」的公平起跑線。原本波動很大的年份被強行「壓弱」，原本平穩的年份被「放大」，這導致 PC1 的主導地位稍微下降（降了 0.52%），變異量被稀釋到了後面的 PC3、PC4 等主成分中。

---

#### 🎯 結論與實務建議：你該選哪一個？

既然單位的級距相同，**「共變異數矩陣 (Covariance / Unscaled)」是更符合公共衛生或流行病學邏輯的選擇。**

* **理由：** 發生率的「真實波動」本身就是非常重要的訊號。如果某個年份或地區的肺癌率在大暴動，我們應該讓 PCA 去捕捉這個真實的變異，而不是用標準化把這個劇烈波動「抹平」。
* **好處：** 誠如你的結果所示，使用共變異數矩陣，你只需要前兩個主成分（PC1 + PC2）就能解釋高達 **70.16%** 的資料變異，降維效果更強，畫出來的 PCA 散佈圖（Biplot）也會更能反映現實世界的波動！

### Comparison 2 (Corr vs Cov)

做 PCA 時，切換「共變異數矩陣（Covariance）」與「相關係數矩陣（Correlation）」最經典的兩個特徵。

---

## 🔍 問題：為什麼 `pca_corr` 的箭頭長度變整齊了，而 `pca_cov` 卻長短不一？

### 💡 結論：因為 `pca_corr` 強制抹平了各年份的「原始波動度（變異數）」。

這正是「要不要標準化（Scaling）」的核心差異：

### 1. 在 `pca_cov`（共變異數 / 未標準化）中：

箭頭的長度直接反映了該年份在各縣市之間的「原始變異數（波動大小）」。

* 像 2019、2020、2022 這些近年的箭頭**超級長**，代表近年來台灣各縣市的肺癌年齡標準化發生率，彼此之間的**貧富差距（差距絕對值）變得非常大**。
* 早期（如 1993 年）的箭頭比較短，代表當年的發生率在各縣市之間相對接近、波動較小。

### 2. 在 `pca_corr`（相關係數 / 標準化後）中：

你的程式碼使用了 `StandardScaler()`，這會把每一年份的原始變異數通通除以她們自己的標準差，強制讓每一年的總變異數都**等於 1**。

* 既然在多維空間裡，每一年的向量總長度都被強迫定為 1，那麼她們投影到 PC1 和 PC2 二維畫布上的長度，自然就會變得非常接近、整齊。
* 此時箭頭的長度不再代表波動大小，而是代表「這兩個人（PC1 和 PC2）對該年份的解釋能力（$R^2$ 或相關係數）」。因為 PC1+PC2 解釋了高達 67.18% 的總變異，絕大多數年份都被解釋得很好，所以箭頭長度看起來就差不多。

---

### 🎨 總結你的兩張圖

* 如果你想強調「近年因為某些因素，各縣市的肺癌發生率差距（絕對數值）正在劇烈擴大」 $\rightarrow$ 請用 **`pca_cov`**（長短不一的箭頭非常有故事性）。
* 如果你想強調「不論早年還是近年，各年份跟主成分的相關性結構都很穩定」 $\rightarrow$ 請用 **`pca_corr`**（整齊的箭頭更具結構感）。

## K-Means

#### 手肘法 (Elbow Method)

這個方法是去計算每個點到其所屬群集中心的距離平方和（SSE，在 sklearn 中稱為 inertia_）。

當 $k$ 越大，SSE 一定會越小。我們要找的是「下降斜率突然變平緩的那個轉折點」（就像手肘一樣）。

In [ ]:
sse = []
k_range = range(1, 11)
for i in k_range:
    km = KMeans(n_clusters=i, random_state=42, n_init=10)
    km.fit(scaled_data)
    sse.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, sse, marker='o', color='#2b5c8f', linewidth=2)
plt.title('Elbow Method For Optimal k', fontsize=14)
plt.xlabel('Number of clusters (k)')
plt.ylabel('SSE (Inertia)')
plt.xticks(k_range)
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig('kmeans_elbow_plot.png', dpi=300)
plt.show()

In [ ]:
sse

#### 輪廓分數（Silhouette Score）

衡量的是「群體內的凝聚度」與「群體間的分離度」。

分數範圍在 $-1$ 到 $+1$ 之間：

* 分數越接近 1：代表分群非常完美，同群的點很近，異群的點很遠。
* 分數接近 0：代表群體邊界重疊，分得不夠好。

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []
k_range = range(2, 11) # 輪廓分數必須從 k=2 開始計算
for i in k_range:
    km = KMeans(n_clusters=i, random_state=42, n_init=10)
    labels = km.fit_predict(scaled_data)
    silhouette_scores.append(silhouette_score(scaled_data, labels))

plt.figure(figsize=(8, 5))
plt.plot(k_range, silhouette_scores, marker='s', color='#e06666', linewidth=2)
plt.title('Silhouette Score For Optimal k', fontsize=14)
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range)
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig('kmeans_silhouette_plot.png', dpi=300)
plt.show()

In [ ]:
silhouette_scores

#### $k$ 要選多少？

結合手肘法的 SSE 降幅與輪廓分數（Silhouette Score），最推薦的 $k$ 值是 **$k=2$**。如果一定要切得更細，第二候選人則是 **$k=5$**。而原本設定的 $k=4$ 雖然在統計上不是最高分，但在公衛實務上依然有它的解釋策略。

以下幫你做精準的數據拆解與決策分析：

---

### 1. 輪廓分析法（Silhouette Score）—— 最直觀的勝負

輪廓分數是「群體凝聚力」與「分離度」的指標，分數越高越好。你的數據從 $k=2$ 開始（因為分 1 群無法計算輪廓）：

| 分群數量 ($k$) | 輪廓分數 (Silhouette Score) | 評語 |
| --- | --- | --- |
| **$k=2$** | **0.3486** 🌟 (最高分) | **統計上的最佳解**。群體邊界最清晰。 |
| $k=3$ | 0.2436 | 分數大幅下滑，代表分 3 群會讓邊界變得模糊。 |
| $k=4$ | 0.2252 | 跌到谷底，說明分 4 群時，群體之間有嚴重的重疊。 |
| **$k=5$** | **0.2523** 📈 (次高分) | 出現了小反彈！代表分 5 群的結構比 3 或 4 群更好。 |

---

### 2. 手肘法（Elbow Method）—— 觀察邊際效益

我們來看每一增加一個分群，SSE（誤差平方和）的「下降彈藥」還剩多少：

* **$k=1 \rightarrow 2$**：下殺 **349.3**（降幅驚人，訊號極強）
* **$k=2 \rightarrow 3$**：下殺 **110.7**（跌勢開始明顯減緩）
* **$k=3 \rightarrow 4$**：下殺 **104.9**
* **$k=4 \rightarrow 5$**：下殺 **77.3**
* **$k=5 \rightarrow 6$**：下殺 **44.5**（之後跌幅都卡在 20~30 左右，進入高原期）

從這裡可以看出，最大的「手肘轉折點」其實就發生在 **$k=2$**。過了 $k=2$ 之後，斜率就開始轉為平緩；而另一個微小的轉折則在 **$k=5$** 之後進一步攤平。

---

## 🎯 最終挑選建議

### 方案 A：純統計學最推薦 $\rightarrow$ 選擇 $k=2$

* **理由**：不論是手肘法的第一轉折點，還是輪廓分數的最高峰（`0.3486`），通通指向 $k=2$。
* **公衛故事**：這對應了你前面 PC1 抓出來的核心概念——直接把台灣縣市一刀切成兩大類：**「長期肺癌高盛行區」vs「長期肺癌低盛行區」**（像是把金門縣與中南部少數低風險縣市歸為一類，其餘本島高盛行縣市歸為另一類）。這張圖會非常乾淨漂亮。

### 方案 B：如果想呈現更細緻的歷史軌跡 $\rightarrow$ 選擇 $k=5$

* **理由**：它是 $k=2$ 之外的次高分（`0.2523`），且此時 SSE 降幅（`77.3`）剛好要進入下一個高原期。
* **公衛故事**：分 5 群通常能完美對應你的 PCA 二維空間的「四個象限 + 中央平均區」。你可以把縣市細分為：高盛行/早發型、高盛行/後發型、低盛行/早發型、低盛行/後發型、以及全國平均型。這在政策擬定上會比 $k=2$ 更有深度。

#### 比較 $$k=2,3,4,5$$

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# ==========================================
# 🎯 在這裡自由修改你的 k 值 (例如：2, 3, 4, 5)
k = 4
# ==========================================

# 1. 執行 K-Means 分群
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(scaled_data)

# 2. ✨ 動態生成 Group 名稱清單與對照表 (根據 k 自動產生 A, B, C, D, E...)
# chr(65) 是 'A', chr(66) 是 'B'，以此類推
groups_list = [f'Group {chr(65 + i)}' for i in range(k)]
cluster_map = {i: groups_list[i] for i in range(k)}
encoded_labels = [cluster_map[label] for label in kmeans_labels]

# 3. 建立畫圖用的 DataFrame 與 PCA 降維
counties_names = pivot_table_cleaned.index
pca_kmeans = PCA(n_components=2)
components_kmeans = pca_kmeans.fit_transform(scaled_data)

df_kmeans_plot = pd.DataFrame({
    'PC1': components_kmeans[:, 0],
    'PC2': -components_kmeans[:, 1],  # Y 軸翻轉對齊
    'County': counties_names,
    'Cluster': encoded_labels
})

# 4. 繪圖 (顏色調色盤改用 'Set1'，在 k 較少或到 5 時，視覺對比度都極佳)
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df_kmeans_plot,
    x='PC1',
    y='PC2',
    hue='Cluster',
    hue_order=groups_list,  # ✨ 自動帶入動態群組順序 ['Group A', 'Group B', ...]
    palette='Set1',        # ✨ 改用 'Set1' 高對比調色盤，分 2~5 群都非常漂亮
    s=200,
    alpha=0.9,
    edgecolor='black'
)

# 標註縣市名稱 (中文字)
for i in range(len(df_kmeans_plot)):
    plt.annotate(
        df_kmeans_plot['County'].iloc[i],
        (df_kmeans_plot['PC1'].iloc[i], df_kmeans_plot['PC2'].iloc[i]),
        textcoords='offset points',
        xytext=(0, 12),
        ha='center',
        fontsize=11,
        fontname=prop.get_name() if 'prop' in locals() else None,
        bbox=dict(boxstyle='round,pad=0.2', fc='yellow', alpha=0.3)
    )

# 設定標題和座標軸 (百分比自動抓取現場 PCA 結果)
plt.title(f'K-Means Clustering (k={k})', fontsize=16)
plt.xlabel(f'PC1 ({pca_kmeans.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca_kmeans.explained_variance_ratio_[1]*100:.2f}%)')
plt.grid(True, alpha=0.3)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.legend(title='Clusters', loc='best')

# 💾 儲存圖片 (檔名也會自動帶入 k 值)
plt.savefig(f'kmeans_pca_clusters_k{k}.png', dpi=300, bbox_inches='tight')
plt.show()

# 5. ✨ 動態輸出各群組的縣市名單
print(f"\n--- Cluster Assignments (k={k}) ---")
for group in groups_list:
    members = df_kmeans_plot[df_kmeans_plot['Cluster'] == group]['County'].tolist()
    print(f"{group}: {'、'.join(members)}")

#### 小節

```
--- Cluster Assignments (k=2) ---
Group A: 台中市、台北市、台南市、台東縣、嘉義市、嘉義縣、基隆市、宜蘭縣、彰化縣、新北市、新竹市、桃園市、澎湖縣、花蓮縣、雲林縣、高雄市
Group B: 南投縣、屏東縣、新竹縣、苗栗縣、金門縣
--- Cluster Assignments (k=3) ---
Group A: 台北市、基隆市、新北市
Group B: 台中市、台南市、台東縣、嘉義市、嘉義縣、宜蘭縣、彰化縣、新竹市、桃園市、澎湖縣、花蓮縣、雲林縣、高雄市
Group C: 南投縣、屏東縣、新竹縣、苗栗縣、金門縣
--- Cluster Assignments (k=4) ---
Group A: 台中市、台南市、台東縣、嘉義市、嘉義縣、彰化縣、新竹市、桃園市、澎湖縣、花蓮縣、雲林縣、高雄市
Group B: 金門縣
Group C: 台北市、基隆市、宜蘭縣、新北市
Group D: 南投縣、屏東縣、新竹縣、苗栗縣
--- Cluster Assignments (k=5) ---
Group A: 台中市、台南市、台東縣、嘉義市、嘉義縣、宜蘭縣、彰化縣、新竹市、桃園市、花蓮縣、雲林縣、高雄市
Group B: 金門縣
Group C: 南投縣、屏東縣、新竹縣、苗栗縣
Group D: 台北市、基隆市、新北市
Group E: 澎湖縣
```

這組跑出來的結果簡直太漂亮了！它就像一根剝洋蔥的線索，把台灣幾十年來肺癌發生率的「結構轉變」交代得清清楚楚。

我們可以從這份分群名單中，讀出一個非常精彩的**公衛故事演進**：

---

### 1. 當 $k=2$ 時：切出「北北基」與「農業/離島縣市」的大基調

* **Group B**：南投、屏東、新竹縣、苗栗、金門。
* **解讀**：這也就是你前面 PC1 抓出來的最基本二分法。這 5 個縣市在台灣的肺癌整體嚴重度（橫軸）屬於**長期低盛行區**。其他本島大都市、東部與澎湖，則集體被歸在 Group A（高盛行區）。

### 2. 當 $k=3$ 時：最關鍵的「北北基」獨立！

* **Group A**：**台北市、基隆市、新北市**。
* **解讀**：你看！一開放成 3 群，北北基三兄弟立刻從大群體中「抱團獨立」了。這完全對應了你前面 Biplot 的右下角區塊——她們屬於「全台最資深的肺癌重災區」。早年發生率極高，但近年因為早期篩檢普及與醫療資源集中，曲線已經走平。

### 3. 當 $k=4$ 時：邊緣孤島的異動

* **Group B（金門）** 與 **Group C（北北基宜）**。
* **解讀**：金門縣因為數據實在太特殊（長期全國最低），在 $k=4$ 時終於忍不住自己獨立成一類（Group B）。而原本在 $k=3$ 屬於 B 群的**宜蘭縣**，在此時被歸入了北北基（Group C），這代表宜蘭的肺癌型態其實跟雙北非常相似。

### 4. 當 $k=5$ 時：這份分析最靈魂的發現——「澎湖縣」

* **Group E**：**澎湖縣**。
* **解讀**：這就是為什麼 $k=5$ 的輪廓分數會出現「小反彈」的真正原因！在 $k=5$ 時，**澎湖縣正式獨立建國了**。
* 還記得前面提到 PC2 代表的意義嗎？澎湖縣屬於「極端的後發型」。她早年發生率低到不可思議，近年卻暴風式飆升。因為她的時間演變軌跡跟台灣本島所有縣市都不同，所以當你把分群切到 5 群時，演算法終於幫你把這個「公衛特例」單獨揪了出來。

---

## 📝 最終報告的「故事線」建議

如果你要在報告或論文中呈現這段分析，**強烈建議選擇 $k=5$ 的圖表與名單**。

因為 $k=5$ 的分群結果最具有傳奇色彩，你可以這樣為這 5 個群體（Groups）命名，報告會變得超級生動且專業：

1. **Group D (北北基三兄弟)** $\rightarrow$ **「歷史重災型」**：早年發生率極高，目前進入高原期。
2. **Group A (中南部各大都會與花東)** $\rightarrow$ **「全國平均型」**：最穩定、最符合台灣整體肺癌擴散軌跡的常態群體。
3. **Group C (竹苗中彰投屏)** $\rightarrow$ **「西部平緩型」**：相較於都會區，長期以來維持在中低水準。
4. **Group B (金門縣)** $\rightarrow$ **「世外桃源型」**：數十年來始終維持全國最低的標準化發生率。
5. **Group E (澎湖縣)** $\rightarrow$ **「近年突變型（關鍵警告）」**：早年極低，近年卻急起直追。這正是最需要公衛學者深入研究、介入調查（如微環境、飲食或空污變遷）的重點縣市。

你覺得選 $k=5$ 來寫這個「澎湖獨立」的故事，符不符合你當初研究這份資料的預期呢？

## Hierarchical Clustering

In [ ]:
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.font_manager as fm
import os

# Standardize data - Handling NaNs (e.g., Lienchiang County)
clustering_data_clean = pivot_table_age_std.dropna()

scaler = StandardScaler()
scaled_clustering_data = scaler.fit_transform(clustering_data_clean)

# Calculate Pearson correlation-based distance
pearson_distances = pdist(scaled_clustering_data, metric='correlation')
if not np.all(np.isfinite(pearson_distances)):
    pearson_distances = np.nan_to_num(pearson_distances, nan=1.0)

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 初始化參數
methods = ['single', 'complete', 'average', 'ward']
k_range = range(2, 11)
results = []

# 2. 巢狀迴圈計算分數
for k in k_range:
    for method in methods:
        clusterer = AgglomerativeClustering(n_clusters=k, linkage=method)
        labels = clusterer.fit_predict(scaled_clustering_data)
        score = silhouette_score(scaled_clustering_data, labels)
        results.append({
            'k': k,
            'Method': method.capitalize(),
            'Silhouette Score': score
        })

# 3. 轉換為 DataFrame
df_comparison = pd.DataFrame(results)

# 4. 建立樞紐表以便觀察
df_pivot = df_comparison.pivot(index='k', columns='Method', values='Silhouette Score')
print("--- Silhouette Score Comparison (k=2 to 10) ---")
display(df_pivot)

# 5. 繪製趨勢圖
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_comparison, x='k', y='Silhouette Score', hue='Method', marker='o')
plt.title('Silhouette Score Trends by Linkage Methods', fontsize=14)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(list(k_range))
plt.legend(title='Linkage Method')

# 💾 ✨ 新增：儲存圖片（必須在 plt.show() 之前）
# dpi=300 確保圖片印刷或放入簡報時非常清晰
# bbox_inches='tight' 可以防止標籤或圖例被切到
plt.savefig('HClust_silhouette_score_trends.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
print(df_pivot)

In [ ]:
import pandas as pd
from sklearn.cluster import AgglomerativeClustering

# 1. 準備必要數據
counties_names = pivot_table_cleaned.index
methods = ['single', 'complete', 'average', 'ward']
k_targets = [2, 3, 4, 5]

# 2. 建立基準 DataFrame (以縣市名稱為 Index)
df_cluster_results = pd.DataFrame(index=counties_names)

# 3. 巢狀迴圈跑完 16 種組合，並直接塞入 DataFrame
for method in methods:
    for k in k_targets:
        # 初始化層次聚類模型
        clusterer = AgglomerativeClustering(n_clusters=k, linkage=method)
        labels = clusterer.fit_predict(scaled_clustering_data)

        # 動態生成 Group A, B, C... 的名稱對應
        groups_list = [f'Group {chr(65 + i)}' for i in range(k)]
        encoded_labels = [groups_list[label] for label in labels]

        # 欄位命名規則，例如: Ward_k5, Average_k2
        col_name = f"{method.capitalize()}_k{k}"

        # 寫入大表格中
        df_cluster_results[col_name] = encoded_labels

# 4. 排序欄位，讓表格由 k=2 到 k=5 依序排列，比較好閱讀
ordered_columns = [f"{m.capitalize()}_k{k}" for k in k_targets for m in methods]
df_cluster_results = df_cluster_results[ordered_columns]

# 5. 印出完整大表格
print("=== 4種 Linkage Methods 於 k=2,3,4,5 之實體分群結果大表 ===")
# 使用 pd.option_context 可以確保在 Jupyter 中完整秀出所有欄位不被隱藏
with pd.option_context('display.max_columns', None, 'display.expand_frame_repr', False):
    display(df_cluster_results)

# 6. 自動將這份珍貴的對照表存成 CSV，方便你直接拉進 Excel 做報告
df_cluster_results.to_csv('hierarchical_clustering_comprehensive_results.csv', encoding='utf-8-sig')
print("\n💾 完整對照表已成功儲存為 'hierarchical_clustering_comprehensive_results.csv' !")

#### HClust linkage method & $k$

這份剛出爐的 `hierarchical_clustering_comprehensive_results.csv` 對照大表，裡面的數據藏著非常多有趣的秘密！

我幫你把這張複雜的表做了深入的**橫向與縱向對比分析**。這張表不僅證實了我們之前的統計推論，還揭露了各個演算法在劃分台灣縣市肺癌型態時的「大亂鬥」過程：

---

### 🔍 亮點一：`Average_k5` 與 `Ward_k5` 的「世紀大收斂」

還記得你在上一輪算輪廓分數時，發現 `Average` 和 `Ward` 在 $k=5$ 的分數（`0.252279`）一模一樣嗎？我們直接來看這份實體名單，她們的分群結果**完全同步**：

* **Group B**（竹苗中彰投屏區，也就是西部平緩型）：南投縣、屏東縣、新竹縣、苗栗縣、雲林縣（Ward 把雲林歸在 A，這是極少數的微調）。
* **Group C**（歷史重災區）：台北市、基隆市、新北市。
* **Group E**（近年突變型）：**澎湖縣**。
* **Group D**（世外桃源型）：**金門縣**。
* **Group A**：其餘的中南部都會與花東。

**💡 結論**：這證實了在 $k=5$ 時，台灣縣市的肺癌時空變遷結構非常強固，不管是追求群內變異最小的 `Ward` 還是看平均距離的 `Average`，都看到了完全相同的地理公衛圖像。

---

### 🔍 亮點二：徹底現形的 `Single` 鏈狀效應（Chaining Effect）

看這份名單可以發現，`Single`（單一連結法）在實務上為什麼這麼不合用：

* **在 `Single_k2`、`Single_k3`、`Single_k4` 中**：除了金門縣、台北市等少數極端點被一個一個「蠶食」單獨踢出去之外，絕大多數的縣市（從南投到高雄）從頭到尾都被死死地綁在 **`Group A`** 裡面。
* 它之所以在 $k=2$ 拿到高分（`0.4613`），純粹是因為它把金門（`Group B`）切出去，剩下 21 個縣市集體大鍋炒（`Group A`）。這種分群對公共衛生完全沒有政策擬定的參考價值。

---

### 🔍 亮點三：`Complete`（完全連結法）的獨特視角

`Complete` 法的特性是傾向於切出「直徑差不多大」的圓形群體，它的分群軌跡很有意思：

* 在 **$k=3$** 時：它把全台分成三塊：
1. `Group A`（花蓮、台東、宜蘭等非都會與東部地區）。
2. `Group B`（金門縣）。
3. `Group C`（雙北、基隆、台中、高雄等核心大都會）。


* 它的邏輯跟 `Ward` 的時空演變不同，`Complete` 更有種「都市化程度/生活型態」的切法。

---

## 📝 總結：給報告的最終建議

1. **直接淘汰 `Single` 法**：因為它無法有效分辨本島縣市的異質性。
2. **強烈推薦選用 `Ward` 或 `Average` 的 $k=5$**：因為這兩個方法在此時達成高度共識，而且成功幫你抓出了最具公衛警示意義的「澎湖縣（獨自獨立為 Group E）」**與**「金門縣（Group D）」，同時把本島分成了大都會、西部常態與非都會三群。

這份 CSV 已經幫你把所有的統計論證基礎打得非常死，直接用 $k=5$ 的 Ward 成果來寫你的故事，這篇報告的邏輯會非常無懈可擊！

#### Dendrogram

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.cluster.hierarchy import linkage, dendrogram

# 1. 準備必要數據 (請確保此時使用的數據為歐氏距離專用的標準化資料)
county_names = pivot_table_cleaned.index.tolist()
linkage_methods = ['single', 'complete', 'average', 'ward']

# 中文字型安全設定
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# 2. 開始依序繪製四張獨立的樹狀圖
for method in linkage_methods:
    plt.figure(figsize=(13, 7))

    # 現場計算歐氏距離的階層結合矩陣 Z
    Z = linkage(scaled_clustering_data, method=method, metric='euclidean')

    # 3. 🎯 依據你的統計表格，動態計算對應的切線高度與染色門檻
    # Z[-1, 2] 是 2->1 群的高度, Z[-2, 2] 是 3->2 群的高度，以此類推
    h_k2 = (Z[-1, 2] + Z[-2, 2]) / 2  # k=2 的中點切線
    h_k4 = (Z[-3, 2] + Z[-4, 2]) / 2  # k=4 的中點切線
    h_k5 = (Z[-4, 2] + Z[-5, 2]) / 2  # k=5 的中點切線

    # 設定指定的染色門檻 (color_threshold)
    if method == 'ward':
        current_threshold = h_k5  # Ward 法染色切在 5 群
        second_k_label = f'2nd Best: k=5 (Height: {h_k5:.2f})'
        second_k_height = h_k5
    else:
        current_threshold = h_k4  # Single, Complete, Average 染色切在 4 群
        second_k_label = f'2nd Best: k=4 (Height: {h_k4:.2f})'
        second_k_height = h_k4

    # 4. 繪製樹狀圖本體
    dendrogram(
        Z,
        labels=county_names,
        color_threshold=current_threshold,  # 套用動態指定的群數染色
        orientation='top',
        leaf_rotation=45,
        leaf_font_size=11,
        distance_sort='descending'
    )

    # 5. 🗺️ 繪製兩條決策輔助切線
    # 🔴 畫出最高分 (k=2) 的紅色實線
    plt.axhline(y=h_k2, color='#d9383a', linestyle='-', linewidth=2.0,
                label=f'Best: k=2 (Height: {h_k2:.2f})')

    # 🔵 畫出第二高分 (k=4 或 k=5) 的藍色虛線
    plt.axhline(y=second_k_height, color='#2a7ae2', linestyle='--', linewidth=1.8,
                label=second_k_label)

    # 6. 圖表細節美化
    plt.title(f'Hierarchical Clustering Dendrogram (Euclidean - {method.capitalize()} Linkage)',
              fontsize=15, pad=15, fontweight='bold')
    plt.xlabel('Counties / Cities', fontsize=12)
    plt.ylabel('Distance Threshold', fontsize=12)
    plt.grid(axis='y', linestyle=':', alpha=0.5)
    plt.legend(title='Silhouette Decisions', loc='upper left', fontsize=10)
    plt.tight_layout()

    # 7. 💾 自動儲存高解析度圖片 (檔名包含對應的 linkage method)
    plt.savefig(f'dendrogram_euclidean_{method}_two_cutoffs.png', dpi=300, bbox_inches='tight')
    plt.show()

### Map visualization

In [ ]:
import geopandas as gpd
import os
from osgeo import gdal

# Set GDAL config option to restore or create missing .shx file
# Moved outside the try block to ensure it always takes effect early.
gdal.SetConfigOption('SHAPE_RESTORE_SHX', 'YES')

# 1. Define potential paths for the shapefile
shp_paths = [
    '/content/drive/MyDrive/多變量分析/直轄市、縣(市)界線1140318/COUNTY_MOI_1140318.shp',
    '/content/drive/MyDrive/1142課程/多變量分析/直轄市、縣(市)界線1140318/COUNTY_MOI_1140318.shp'
]
found_shp_file_path = None
for path in shp_paths:
    if os.path.exists(path):
        found_shp_file_path = path
        break

if found_shp_file_path:
    try:
        # Load the shapefile and project to WGS84
        taiwan_map = gpd.read_file(found_shp_file_path).to_crs(epsg=4326)
        # Dynamically find the county name column
        county_col = 'COUNTYNAME' if 'COUNTYNAME' in taiwan_map.columns else \
                     [c for c in taiwan_map.columns if 'NAME' in c.upper()][0]
        print(f"Shapefile loaded successfully from: {found_shp_file_path}")
        print(f"Using county column: {county_col}")
        # display(taiwan_map.head()) # Uncomment for inspection if needed
    except Exception as e:
        print(f"An error occurred while reading the shapefile: {e}")
        taiwan_map = None # Ensure taiwan_map is None on error
        county_col = None
else:
    print("Error: The shapefile was not found in any of the specified paths. Please ensure Google Drive is mounted and the file exists.")
    taiwan_map = None # Ensure taiwan_map is None on error
    county_col = None

#### K-Means

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.patches import Patch
from sklearn.cluster import KMeans

# 排除連江縣
taiwan_map_clean = taiwan_map[taiwan_map[county_col] != '連江縣'].copy()
taiwan_map_clean[county_col] = taiwan_map_clean[county_col].str.replace('臺', '台', 'one' if hasattr(str, 'replace_old') else -1)

# 確保沿用你先前的字型設定
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# 確保縣市名稱清單 (對齊 K-means 順序)
counties_names = pivot_table_cleaned.index

# 固定群組顏色
color_dict = {
    'Group A': '#e41a1c',  # 紅
    'Group B': '#377eb8',  # 藍
    'Group C': '#4daf4a',  # 綠
    'Group D': '#984ea3',  # 紫
    'Group E': '#ff7f00'   # 橘
}

# ==========================================
# 2. 進入迴圈，分別繪製 k=2 與 k=5
# ==========================================
for k_val in [2, 5]:

    # 執行 K-Means 分群
    kmeans = KMeans(n_clusters=k_val, random_state=42, n_init=10)
    labels = kmeans.fit_predict(scaled_data)

    # 動態生成群組標籤
    groups_list = [f'Group {chr(65 + i)}' for i in range(k_val)]
    cluster_map = {i: groups_list[i] for i in range(k_val)}
    encoded_labels = [cluster_map[l] for l in labels]

    # 建立分群對照 DataFrame
    df_clusters = pd.DataFrame({
        'County': counties_names,
        'Cluster': encoded_labels
    })

    # 合併地圖與分群數據
    map_data = taiwan_map_clean.merge(df_clusters, left_on=county_col, right_on='County', how='left')
    map_data['color'] = map_data['Cluster'].map(color_dict).fillna('#cbd5e1')

    # ==========================================
    # 3. 開始建構圖表與子母畫面
    # ==========================================
    fig, ax_main = plt.subplots(figsize=(10, 12))

    # 🗺️ (A) 繪製主圖
    map_data.plot(ax=ax_main, color=map_data['color'], edgecolor='black', linewidth=0.4)

    # ✨ 關鍵修改：將經度左界從 120.0 放寬到 119.0，為左側小地圖拉出完美腹地
    ax_main.set_xlim(119.0, 122.3)
    ax_main.set_ylim(21.8, 25.4)

    ax_main.set_title(f'Taiwan K-Means Map\n(k={k_val})', fontsize=16, pad=20, fontweight='bold')
    ax_main.set_xlabel('Longitude (經度)', fontsize=11)
    ax_main.set_ylabel('Latitude (緯度)', fontsize=11)
    ax_main.grid(True, linestyle='--', alpha=0.3)

    for idx, row in map_data[map_data[county_col] != '金門縣'].iterrows():
        if pd.notnull(row['Cluster']):
            centroid = row['geometry'].centroid
            ax_main.annotate(text=row['County'], xy=(centroid.x, centroid.y), ha='center', fontsize=12,
                             bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6, ec='none'))

    # 🗺️ (B) 嵌入金門小地圖
    # ✨ 改用 ax_main.inset_axes，[左邊距, 下邊距, 寬度, 高度] 比例完全鎖在主圖內
    ax_kinmen = ax_main.inset_axes([0.05, 0.72, 0.22, 0.22])
    kinmen_data = map_data[map_data[county_col] == '金門縣']
    kinmen_data.plot(ax=ax_kinmen, color=kinmen_data['color'], edgecolor='black', linewidth=0.5)
    ax_kinmen.set_xlim(118.1, 118.6)
    ax_kinmen.set_ylim(24.2, 24.6)
    ax_kinmen.set_title('金門縣', fontsize=12, fontweight='bold', color='#1e293b')
    ax_kinmen.set_xticks([])
    ax_kinmen.set_yticks([])

    # ==========================================
    # 4. 手動客製化圖例 (Legend)
    # ==========================================
    legend_elements = [
        Patch(facecolor=color_dict[group], edgecolor='black', linewidth=0.5, label=group)
        for group in groups_list
    ]
    ax_main.legend(handles=legend_elements, title='Clusters', loc='lower right', fontsize=12, title_fontsize=13)

    # 💾 儲存高解析度地圖
    plt.savefig(f'map_kmeans_k{k_val}.png', dpi=300, bbox_inches='tight')
    plt.show()

print("\n🎉 地圖（k=2, k=5）已繪製完成")

#### H-Clustering

In [ ]:
# 讀取上一階段打包好的 16 種組合分群結果 (以縣市名稱為 Index)
df_hierarchical = pd.read_csv('hierarchical_clustering_comprehensive_results.csv', index_col=0)

# 確保沿用你的中文字型設定
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# 固定群組高對比顏色面版
color_dict = {
    'Group A': '#e41a1c',  # 紅
    'Group B': '#377eb8',  # 藍
    'Group C': '#4daf4a',  # 綠
    'Group D': '#984ea3',  # 紫
    'Group E': '#ff7f00'   # 橘
}

# ==========================================
# 2. 定義方案 A 與方案 B 的地圖任務清單
# 欄位命名規則必須完美對齊 CSV 欄位（例如：'Single_k2', 'Ward_k5'）
# ==========================================
map_tasks = [
    # 方案 A：所有方法皆看 k = 2
    {'scenario': 'A', 'method': 'Single',   'k': 2},
    {'scenario': 'A', 'method': 'Complete', 'k': 2},
    {'scenario': 'A', 'method': 'Average',  'k': 2},
    {'scenario': 'A', 'method': 'Ward',     'k': 2},

    # 方案 B：指定最佳/次佳群數 (前三者 k=4, Ward k=5)
    {'scenario': 'B', 'method': 'Single',   'k': 4},
    {'scenario': 'B', 'method': 'Complete', 'k': 4},
    {'scenario': 'B', 'method': 'Average',  'k': 4},
    {'scenario': 'B', 'method': 'Ward',     'k': 5},
]

# ==========================================
# 3. 進入自動化自動繪圖迴圈 (共 8 張圖)
# ==========================================
for task in map_tasks:
    scenario = task['scenario']
    method = task['method']
    k_val = task['k']

    # 對齊 CSV 的欄位名稱
    csv_col_name = f"{method}_k{k_val}"

    # 建立暫時的對照 DataFrame 準備進行 Merge
    df_temp = pd.DataFrame({
        'County': df_hierarchical.index,
        'Cluster': df_hierarchical[csv_col_name]
    })

    # 地圖與數據結合
    map_data = taiwan_map_clean.merge(df_temp, left_on=county_col, right_on='County', how='left')
    map_data['color'] = map_data['Cluster'].map(color_dict).fillna('#cbd5e1')

    # 建立畫布
    fig, ax_main = plt.subplots(figsize=(10, 12))

    # 🗺️ (A) 繪製主圖 (範圍包含台灣本島與澎湖縣)
    map_data.plot(ax=ax_main, color=map_data['color'], edgecolor='black', linewidth=0.3)
    ax_main.set_xlim(119.0, 122.3)  # 完美容納澎湖的寬闊經度
    ax_main.set_ylim(21.8, 25.4)

    # 動態精美標題
    ax_main.set_title(f'Taiwan Hierarchical Clustering Map\n{method} Linkage (k={k_val})',
                      fontsize=15, pad=15, fontweight='bold')
    ax_main.set_xlabel('Longitude (經度)', fontsize=11)
    ax_main.set_ylabel('Latitude (緯度)', fontsize=11)
    ax_main.grid(True, linestyle='--', alpha=0.3)

    for idx, row in map_data[map_data[county_col] != '金門縣'].iterrows():
        if pd.notnull(row['Cluster']):
            centroid = row['geometry'].centroid
            ax_main.annotate(text=row['County'], xy=(centroid.x, centroid.y), ha='center', fontsize=12,
                             bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6, ec='none'))

    # 🗺️ (B) 僅保留金門小地圖 (利用 inset_axes 舒適安放於左上角海域)
    ax_kinmen = ax_main.inset_axes([0.05, 0.72, 0.22, 0.22])
    kinmen_data = map_data[map_data[county_col] == '金門縣']
    kinmen_data.plot(ax=ax_kinmen, color=kinmen_data['color'], edgecolor='black', linewidth=0.4)
    ax_kinmen.set_xlim(118.1, 118.6)
    ax_kinmen.set_ylim(24.2, 24.6)
    ax_kinmen.set_title('金門縣', fontsize=12, fontweight='bold', color='#1e293b')
    ax_kinmen.set_xticks([])
    ax_kinmen.set_yticks([])

    # ==========================================
    # 4. 手動客製化圖例 (僅顯示該 k 值範圍內的 Group)
    # ==========================================
    groups_list = [f'Group {chr(65 + i)}' for i in range(k_val)]
    legend_elements = [
        Patch(facecolor=color_dict[group], edgecolor='black', linewidth=0.5, label=group)
        for group in groups_list
    ]
    ax_main.legend(handles=legend_elements, title=f'Clusters (k={k_val})', loc='lower right', fontsize=12, title_fontsize=13)

    # 💾 5. 自動儲存高解析度圖片，檔名格式：map_hierarchical_ScenarioA_Single_k2.png
    file_name = f'map_hierarchical_{method}_k{k_val}.png'
    plt.savefig(file_name, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"📸 已成功儲存地圖：{file_name}")

print("\n✨ 地圖已全部繪製並儲存完畢。")

### Lineplots by Clusters

In [ ]:
# 1. 確保已剔除連江縣，並建立包含分群標籤的 DataFrame
df_plot_raw = pivot_table_age_std.drop(index='連江縣', errors='ignore').copy()

# 假設 df_scores 是我們上一步做好的 DataFrame (內含 County 和 Cluster 欄位)
# 將分群資訊 mapping 回原始的 pivot table
df_plot_raw = df_plot_raw.reset_index()
df_plot_raw = df_plot_raw.merge(df_scores[['County', 'Cluster']], left_on='City_County', right_on='County')

# 2. 將資料由寬轉長 (Melt)，方便 Seaborn 畫折線圖
df_long = df_plot_raw.drop(columns=['County']).melt(
    id_vars=['City_County', 'Cluster'],
    var_name='Year',
    value_name='Incidence_Rate'
)
df_long['Year'] = df_long['Year'].astype(int)

#### K-Means

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

# 排除連江縣 (確保與地圖一致)
df_long_clean = df_long[df_long['City_County'] != '連江縣'].copy()

# ==========================================
# 1. 動態計算 K-Means 標籤 (k=2 與 k=5)
# ==========================================
# K=2 分群
km_2 = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_k2 = km_2.fit_predict(scaled_data)
map_k2 = {i: f'Group {chr(65 + i)}' for i in range(2)}
df_label_k2 = pd.DataFrame({'County': pivot_table_cleaned.index, 'Cluster_k2': [map_k2[l] for l in labels_k2]})

# K=5 分群 (✨ 已依需求由 k=4 改為 k=5)
km_5 = KMeans(n_clusters=5, random_state=42, n_init=10)
labels_k5 = km_5.fit_predict(scaled_data)
map_k5 = {i: f'Group {chr(65 + i)}' for i in range(5)}
df_label_k5 = pd.DataFrame({'County': pivot_table_cleaned.index, 'Cluster_k5': [map_k5[l] for l in labels_k5]})

# 將分群標籤合併回長資料 (Long format)
df_plot_data = df_long_clean.merge(df_label_k2, left_on='City_County', right_on='County', how='left')
df_plot_data = df_plot_data.merge(df_label_k5, left_on='City_County', right_on='County', how='left')

# ==========================================
# 2. 定義統一的主色彩字典 (擴充至 Group E)
# ==========================================
color_dict = {
    'Group A': '#e41a1c',  # 紅
    'Group B': '#377eb8',  # 藍
    'Group C': '#4daf4a',  # 綠
    'Group D': '#984ea3',  # 紫
    'Group E': '#ff7f00'   # 橘 (✨ k=5 新增)
}

# 中文字型安全設定
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# ==========================================
# 3. 繪製圖表一：K-Means (k=2) - 獨立畫布
# ==========================================
k_val = 2
df_means_k2 = df_plot_data.groupby(['Year', 'Cluster_k2'])['Incidence_Rate'].mean().reset_index()

fig1, ax1 = plt.subplots(figsize=(14, 8))

# A. 背景：各縣市極淡細線
sns.lineplot(data=df_plot_data, x='Year', y='Incidence_Rate', hue='Cluster_k2', units='City_County',
             estimator=None, palette=color_dict, linewidth=0.8, alpha=0.15, legend=False, ax=ax1)

# B. 前景：群組平均粗線 (✨ 已採用你的小小更動設定)
sns.lineplot(data=df_means_k2, x='Year', y='Incidence_Rate', hue='Cluster_k2',
             palette=color_dict, linewidth=2.5, marker='o', markersize=5.5, ax=ax1)

# C. 標準化介面與圖例
ax1.set_title(f'各分群年度平均肺癌發生率：K-Means (k={k_val})', fontsize=14, fontweight='bold', pad=15)
ax1.set_ylabel('平均發生率 (每10萬人口)', fontsize=11)
ax1.set_xlabel('診斷年份 (Year)', fontsize=11)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(title=f'Cluster (k={k_val})', loc='upper left', framealpha=0.8)

plt.tight_layout()
plt.savefig(f'trends_kmeans_k{k_val}.png', dpi=300, bbox_inches='tight')
plt.show()


# ==========================================
# 4. 繪製圖表二：K-Means (k=5) - 獨立畫布 (✨ 全新獨立繪製)
# ==========================================
k_val = 5
df_means_k5 = df_plot_data.groupby(['Year', 'Cluster_k5'])['Incidence_Rate'].mean().reset_index()

fig2, ax2 = plt.subplots(figsize=(14, 8))

# A. 背景：各縣市極淡細線
sns.lineplot(data=df_plot_data, x='Year', y='Incidence_Rate', hue='Cluster_k5', units='City_County',
             estimator=None, palette=color_dict, linewidth=0.8, alpha=0.15, legend=False, ax=ax2)

# B. 前景：群組平均粗線 (✨ 同步套用你的更動設定)
sns.lineplot(data=df_means_k5, x='Year', y='Incidence_Rate', hue='Cluster_k5',
             palette=color_dict, linewidth=2.5, marker='o', markersize=5.5, ax=ax2)

# C. 標準化介面與圖例
ax2.set_title(f'各分群年度平均肺癌發生率：K-Means (k={k_val})', fontsize=14, fontweight='bold', pad=15)
ax2.set_ylabel('平均發生率 (每10萬人口)', fontsize=11)
ax2.set_xlabel('診斷年份 (Year)', fontsize=11)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(title=f'Cluster (k={k_val})', loc='upper left', framealpha=0.8)

plt.tight_layout()
plt.savefig(f'trends_kmeans_k{k_val}.png', dpi=300, bbox_inches='tight')
plt.show()


# ==========================================
# 5. 數據摘要輸出 (✨ 擴充同時顯示 k=2 與 k=5 的結果)
# ==========================================
print("\n" + "="*50)
print("   📊 肺癌平均發生率增長摘要 (1979-2022)   ")
print("="*50)

# 輸出 K=2 摘要
print("\n🔹 【K-Means 分群結果 (k=2)】")
for group in sorted(df_means_k2['Cluster_k2'].unique()):
    subset = df_means_k2[df_means_k2['Cluster_k2'] == group]
    start, end = subset.iloc[0]['Incidence_Rate'], subset.iloc[-1]['Incidence_Rate']
    pct = ((end - start) / start) * 100
    print(f"  - {group}: {start:.2f} -> {end:.2f} ({pct:+.1f}%)")

# 輸出 K=5 摘要
print("\n🔸 【K-Means 分群結果 (k=5)】")
for group in sorted(df_means_k5['Cluster_k5'].unique()):
    subset = df_means_k5[df_means_k5['Cluster_k5'] == group]
    start, end = subset.iloc[0]['Incidence_Rate'], subset.iloc[-1]['Incidence_Rate']
    pct = ((end - start) / start) * 100
    print(f"  - {group}: {start:.2f} -> {end:.2f} ({pct:+.1f}%)")
print("\n" + "="*50)

K-Means 分群的增長摘要


```
==================================================
   📊 肺癌平均發生率增長摘要 (1979-2022)   
==================================================
🔹 【K-Means 分群結果 (k=2)】
  - Group A: 11.05 -> 42.22 (+282.1%)
  - Group B: 5.00 -> 32.23 (+544.2%)

🔸 【K-Means 分群結果 (k=5)】
  - Group A: 9.77 -> 41.90 (+328.8%)
  - Group B: 2.40 -> 16.92 (+605.0%) (金門)
  - Group C: 5.66 -> 36.06 (+537.7%)
  - Group D: 19.42 -> 41.86 (+115.5%)
  - Group E: 1.27 -> 47.21 (+3617.3%) (澎湖)
==================================================
```



#### H-Clustering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 資料清理與安全對齊 (Data Preparing)
# ==========================================
# 排除連江縣 (確保與地圖、K-Means 的基底完全一致)
df_long_clean = df_long[df_long['City_County'] != '連江縣'].copy()

# 讀取先前儲存的 16 種組合層次聚類結果 (以縣市名稱為 Index)
df_hierarchical = pd.read_csv('hierarchical_clustering_comprehensive_results.csv', index_col=0)

# ==========================================
# 2. 定義統一的主色彩字典 (與地圖、K-Means 保持絕對一致)
# ==========================================
color_dict = {
    'Group A': '#e41a1c',  # 紅
    'Group B': '#377eb8',  # 藍
    'Group C': '#4daf4a',  # 綠
    'Group D': '#984ea3',  # 紫
    'Group E': '#ff7f00'   # 橘
}

# 中文字型安全設定 (沿用您先前的設定)
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# ==========================================
# 3. 配置方案 A 與方案 B 的繪圖任務清單 (Task Configuration)
# ==========================================
map_tasks = [
    # 方案 A：所有方法皆看 k = 2
    {'scenario': 'A', 'method': 'Single',   'k': 2},
    {'scenario': 'A', 'method': 'Complete', 'k': 2},
    {'scenario': 'A', 'method': 'Average',  'k': 2},
    {'scenario': 'A', 'method': 'Ward',     'k': 2},

    # 方案 B：指定最佳/次佳群數 (前三者 k=4, Ward k=5)
    {'scenario': 'B', 'method': 'Single',   'k': 4},
    {'scenario': 'B', 'method': 'Complete', 'k': 4},
    {'scenario': 'B', 'method': 'Average',  'k': 4},
    {'scenario': 'B', 'method': 'Ward',     'k': 5},
]

# 建立一個字典，用來動態收集各任務的摘要數據，最後統一漂亮輸出
summary_logs = {}

# ==========================================
# 4. 進入自動化繪圖迴圈 (共 8 張圖獨立畫布)
# ==========================================
for task in map_tasks:
    scenario = task['scenario']
    method = task['method']
    k_val = task['k']

    # 對齊 CSV 的分群標籤欄位名稱（例如：'Single_k2', 'Ward_k5'）
    csv_col_name = f"{method}_k{k_val}"

    # 建立暫時的對照 DataFrame
    df_temp = pd.DataFrame({
        'County': df_hierarchical.index,
        'Cluster_Label': df_hierarchical[csv_col_name]
    })

    # 將該演算法的分群標籤動態合併進長資料中
    df_task_data = df_long_clean.merge(df_temp, left_on='City_County', right_on='County', how='left')

    # 即時計算該組合在各年度的「群組平均發生率 (Cluster Means)」
    df_task_means = df_task_data.groupby(['Year', 'Cluster_Label'])['Incidence_Rate'].mean().reset_index()

    # 儲存摘要數據供後續輸出
    task_key = f"Scenario {scenario}: {method} Linkage (k={k_val})"
    summary_logs[task_key] = df_task_means

    # ------------------------------------------
    # 建立獨立畫布 (每一張圖都是全新的獨立 Figure)
    # ------------------------------------------
    fig, ax = plt.subplots(figsize=(14, 8))

    # 層次一：背景（所有個別縣市的「極淡細線」）
    # legend=False 確保個別縣市名稱不會污染圖例
    sns.lineplot(data=df_task_data, x='Year', y='Incidence_Rate', hue='Cluster_Label', units='City_County',
                 estimator=None, palette=color_dict, linewidth=0.8, alpha=0.15, legend=False, ax=ax)

    # 層次二：前景（群組平均的「飽和粗線」+ 顯眼轉折圓點）
    # 這裡採用了您的黃金比例參數修正：linewidth=2.5, markersize=5.5
    sns.lineplot(data=df_task_means, x='Year', y='Incidence_Rate', hue='Cluster_Label',
                 palette=color_dict, linewidth=2.5, marker='o', markersize=5.5, ax=ax)

    # ------------------------------------------
    # 5. 介面與圖例標準化設定
    # ------------------------------------------
    # 標題標準化：各分群年度平均肺癌發生率：Hierarchical - {Method} (k={k})
    ax.set_title(f'各分群年度平均肺癌發生率：Hierarchical - {method} (k={k_val})',
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_ylabel('平均發生率 (每10萬人口)', fontsize=11)
    ax.set_xlabel('診斷年份 (Year)', fontsize=11)
    ax.grid(True, linestyle=':', alpha=0.6)

    # 圖例標準化：標題設為 Cluster (k=k_val)
    ax.legend(title=f'Cluster (k={k_val})', loc='upper left', framealpha=0.8)

    # 檔案命名標準化並安全儲存
    file_name = f'trend_hierarchical_{method}_k{k_val}.png'
    plt.tight_layout()
    plt.savefig(file_name, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"📸 獨立趨勢圖已成功儲存：{file_name}")

# ==========================================
# 6. 數據摘要動態輸出 (Growth Summary Logging)
# ==========================================
print("\n" + "="*60)
print("   📊 層次聚類分群：肺癌平均發生率增長摘要 (1979-2022)   ")
print("="*60)

for task_title, df_means in summary_logs.items():
    print(f"\n🔹 【{task_title}】")
    for group in sorted(df_means['Cluster_Label'].unique()):
        subset = df_means[df_means['Cluster_Label'] == group]

        # 安全機制：防範某些極端演算法分群中出現空資料
        if not subset.empty:
            start, end = subset.iloc[0]['Incidence_Rate'], subset.iloc[-1]['Incidence_Rate']
            pct = ((end - start) / start) * 100
            print(f"  - {group}: {start:.2f} -> {end:.2f} ({pct:+.1f}%)")
        else:
            print(f"  - {group}: 無資料")

print("\n" + "="*60)
print("🎉 所有層次聚類學術折線圖（共 8 張）與增長統計摘要已全部生成完畢！")

Hierarchical Clustering 的增長摘要

```
============================================================
   📊 階層式分群：肺癌平均發生率增長摘要 (1979-2022)   
============================================================
🔹 【Scenario A: Single Linkage (k=2)】
  - Group A: 9.97 -> 40.99 (+311.1%)
  - Group B: 2.40 -> 16.92 (+605.0%)

🔹 【Scenario A: Complete Linkage (k=2)】
  - Group A: 9.97 -> 40.99 (+311.1%)
  - Group B: 2.40 -> 16.92 (+605.0%)

🔹 【Scenario A: Average Linkage (k=2)】
  - Group A: 9.97 -> 40.99 (+311.1%)
  - Group B: 2.40 -> 16.92 (+605.0%)

🔹 【Scenario A: Ward Linkage (k=2)】
  - Group A: 11.05 -> 42.22 (+282.1%)
  - Group B: 5.00 -> 32.23 (+544.2%)
------------------------------------------------------------
🔹 【Scenario B: Single Linkage (k=4)】
  - Group A: 11.70 -> 41.89 (+258.0%)
  - Group B: 5.66 -> 36.06 (+537.7%)
  - Group C: 1.27 -> 47.21 (+3617.3%)
  - Group D: 2.40 -> 16.92 (+605.0%)

🔹 【Scenario B: Complete Linkage (k=4)】
  - Group A: 12.08 -> 42.58 (+252.4%)
  - Group B: 2.40 -> 16.92 (+605.0%)
  - Group C: 6.84 -> 36.51 (+433.5%)
  - Group D: 1.27 -> 47.21 (+3617.3%)

🔹 【Scenario B: Average Linkage (k=4)】
  - Group A: 11.70 -> 41.89 (+258.0%)
  - Group B: 5.66 -> 36.06 (+537.7%)
  - Group C: 1.27 -> 47.21 (+3617.3%)
  - Group D: 2.40 -> 16.92 (+605.0%)

🔹 【Scenario B: Ward Linkage (k=5)】
  - Group A: 9.77 -> 41.90 (+328.8%)
  - Group B: 5.66 -> 36.06 (+537.7%)
  - Group C: 19.42 -> 41.86 (+115.5%)
  - Group D: 2.40 -> 16.92 (+605.0%)
  - Group E: 1.27 -> 47.21 (+3617.3%)
============================================================
```



### 小節

一、

如果您仔細對比以下兩組數據：

* 【 K-Means (k=2) 】 與 【 Hierarchical: Ward (k=2) 】
* 【 K-Means (k=5) 】 與 【 Hierarchical: Ward (k=5) 】

您會發現它們的起始值、結束值與增長率完全一模一樣！

**解讀**：這在統計學上是一個非常漂亮的實證。因為 Ward's Method（最小變異法） 的核心優化目標就是「最小化群內的平方誤差和（SSE）」，這與 K-Means 的目標函數在數學本質上是完全一致的。這項結果有力地證明了這兩個截然不同的演算法（一個是迭代分割、一個是凝聚層次），在空間分布與趨勢切分上達到了完美的收斂。

二、

看 Scenario A (k=2) 時，Single、Complete、Average 三種聯結法產出的群組數據完全一致：

Group A: 9.97 $\rightarrow$ 40.99

Group B: 2.40 $\rightarrow$ 16.92（這條線其實就是「金門縣」的獨立軌跡）

**解讀**：這揭示了傳統層次聚類在 $k$ 值過小時的經典缺陷——連鎖效應（Chaining Effect）或對極端值的過度敏感。除了金門縣（Group B）因為早期發生率極低且成長平緩被孤立出來之外，全台灣其他 20 個縣市不論本島、外島、南北差異，全部被硬生生塞進了 Group A。這證明了在 $k=2$ 時，這三種方法無法提供有意義的地理宏觀分群，唯獨 Ward 法 成功將台灣切分成均勻的兩大宏觀地理板塊（Group A 與 B）。

---

📝 報告結論寫作建議（Takeaway）

在您的報告中，強烈建議推崇 K-Means (k=5) 與 Hierarchical: Ward Linkage (k=5) 作為本次研究的最佳視覺化與分群模型。

寫作範例：
本研究對比了多種聚類演算法在 1979-2022 年台灣肺癌發生率長軌跡數據上的表現。實驗結果顯示，Single、Complete、Average 聯結法在小群數時極易受到金門、澎湖等外島極端歷史軌跡的干擾，產生嚴重的孤立群組（Chaining Effect），無法有效剖析台灣本島的地理流行病學結構。

相反地，Ward 最小變異法（k=5）與 K-Means（k=5）呈現了完全一致的收斂結果。該模型不僅精準捕捉了增長高達 +3617.3% 的極端突變型縣市（Group E），更成功將台灣本島依據「歷史高基數飽和型（Group C, +115.5%）」、「持續高攀型（Group A）」與「後期爆發型（Group B, +537.7%）」進行了具有高度公衛防癌實政價值的多層次切分，最適合做為後續資源配置的決策依據。

## 附錄

* Some old version code

In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram

# 1. 準備必要數據
county_names = pivot_table_cleaned.index.tolist()

# 2. 計算兩種方法在多維空間中的 Linkage 矩陣
# (Ward 法與 Average 法會依據不同的距離演算法建立階層樹)
Z_ward = linkage(scaled_clustering_data, method='ward')
Z_average = linkage(scaled_clustering_data, method='average')

# 確保全域中文字型已設定
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# ==========================================================
# 📊 圖片一：Ward's Method 樹狀圖
# ==========================================================
plt.figure(figsize=(14, 6))

# ✨ 核心技巧：動態計算 k=5 的顏色切分門檻
# Z 矩陣的第三欄 [-4] 代表倒數第 4 次融合的高度，低於此高度剛好會切出 5 個群體
threshold_ward = Z_ward[-4, 2]

dendrogram(
    Z_ward,
    labels=county_names,
    color_threshold=threshold_ward,  # 自動讓 5 個群體染上不同顏色
    leaf_rotation=45,                # 縣市名稱旋轉 45 度避免重疊
    leaf_font_size=12
)

# 畫出視覺輔助切線
plt.axhline(y=threshold_ward, color='#e06666', linestyle='--', linewidth=1.5,
            label=f'k=5 Cutoff Line (Height: {threshold_ward:.2f})')

plt.title("Hierarchical Clustering Dendrogram (Ward's Method, k=5)", fontsize=16, pad=15)
plt.xlabel('Counties / Cities', fontsize=12)
plt.ylabel('Cluster Distance (Inertia)', fontsize=12)
plt.grid(axis='y', linestyle=':', alpha=0.5)
plt.legend(loc='upper right')
plt.tight_layout()

# 💾 儲存第一張圖
plt.savefig('dendrogram_ward_k5.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ==========================================================
# 📊 圖片二：Average Linkage Method 樹狀圖
# ==========================================================
plt.figure(figsize=(14, 6))

# 同理，計算 Average 法在 k=5 時的切分高度
threshold_avg = Z_average[-4, 2]

dendrogram(
    Z_average,
    labels=county_names,
    color_threshold=threshold_avg,   # 自動讓 5 個群體染上不同顏色
    leaf_rotation=45,
    leaf_font_size=12
)

# 畫出視覺輔助切線
plt.axhline(y=threshold_avg, color='#e06666', linestyle='--', linewidth=1.5,
            label=f'k=5 Cutoff Line (Height: {threshold_avg:.2f})')

plt.title("Hierarchical Clustering Dendrogram (Average Method, k=5)", fontsize=16, pad=15)
plt.xlabel('Counties / Cities', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.grid(axis='y', linestyle=':', alpha=0.5)
plt.legend(loc='upper right')
plt.tight_layout()

# 💾 儲存第二張圖
plt.savefig('dendrogram_average_k5.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv("/content/肺癌罹病率資料.csv")
print(f"CSV file loaded successfully")
display(df.head())

In [ ]:
df_filtered = df[(df['性別'] == '全') &(df['縣市別'] != '全國') & (df['癌症別'] == '肺、支氣管及氣管')]
display(df_filtered)

In [ ]:
pivot_table_age_std = df_filtered.pivot_table(
    index='縣市別',
    columns='癌症診斷年',
    values='年齡標準化發生率  WHO 2000世界標準人口 (每10萬人口)',
    aggfunc='mean' # Assuming you want the mean if there are multiple entries
)
display(pivot_table_age_std)

In [ ]:
pivot_table_age_std.to_csv('pivot_table.csv', na_rep="NA", encoding='utf-8-sig')